# LeetCode 336 — Palindrome Pairs (Hard) bằng Python

Notebook này giảng **từ căn bản** một bài chuỗi rất hay: trong một danh sách từ, tìm **mọi cặp** `(i, j)` sao cho ghép `words[i] + words[j]` được một **palindrome**. Skeleton trên LeetCode (Python3):

```python
class Solution:
    def palindromePairs(self, words: List[str]) -> List[List[int]]:

```

---

## Sau notebook này, bạn sẽ nắm được

1. **Brute force** thử mọi cặp `(i, j)` — cách người mới tự nghĩ ra — và tự tay đo xem nó chậm cỡ nào khi `n = 5000`.
2. **Quan sát then chốt**: một palindrome `A + B` luôn "lộ" ra ở **một trong hai từ** dưới dạng *một khúc palindrome* + *đảo ngược của từ kia*. Từ đó suy ra cách **cắt mỗi từ thành `prefix | suffix`**.
3. **Lời giải chính — hash map (dict) `từ → chỉ số`**: hai luật cắt, cách xử lý **chuỗi rỗng**, **không ghép một từ với chính nó**, và **không sinh cặp trùng** — kèm trace từng lát cắt và chứng minh vì sao không sót, không trùng.
4. **Trie (cây tiền tố)** học từ con số 0, rồi dựng **Trie trên các từ đảo ngược** có đánh dấu "phần còn lại là palindrome".
5. (Nâng cao) Dùng **mảng `lps` của KMP** để biết *mọi* tiền tố/hậu tố palindrome của một từ trong `O(k)` — đạt đúng yêu cầu `O(tổng độ dài các từ)` mà đề đặt ra.
6. Lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và **6 bài tập có đáp án**.

Lộ trình: `Brute force mọi cặp O(n²·k)` → `Hash map + cắt từ O(n·k²)` → `Trie từ đảo ngược O(n·k²)` → `Trie + lps O(tổng độ dài)`.

## 1. Đề bài

Cho một mảng **các chuỗi đôi một khác nhau** `words` (đánh chỉ số từ 0).

Một **cặp palindrome** (palindrome pair) là một cặp số nguyên `(i, j)` thỏa mãn:

- `0 <= i, j < words.length`,
- `i != j`, và
- `words[i] + words[j]` (nối hai chuỗi lại) là một **palindrome** — chuỗi đọc xuôi hay đọc ngược đều giống nhau.

Trả về một mảng chứa **tất cả** các cặp palindrome của `words`.

Đề còn yêu cầu: *You must write an algorithm with `O(sum of words[i].length)` runtime complexity* — thuật toán phải chạy trong thời gian **`O(tổng độ dài các từ)`**. (Mục 13 và 18 sẽ bàn kỹ yêu cầu này: lời giải phổ biến `O(n·k²)` vẫn được chấp nhận trên LeetCode, còn Cách 4 mới thật sự đạt `O(tổng độ dài)`.)

### Ví dụ trong đề

| | Input `words` | Output | Các palindrome tạo được |
|---|---|---|---|
| Ví dụ 1 | `["abcd","dcba","lls","s","sssll"]` | `[[0,1],[1,0],[3,2],[2,4]]` | `"abcddcba"`, `"dcbaabcd"`, `"slls"`, `"llssssll"` |
| Ví dụ 2 | `["bat","tab","cat"]` | `[[0,1],[1,0]]` | `"battab"`, `"tabbat"` |
| Ví dụ 3 | `["a",""]` | `[[0,1],[1,0]]` | `"a"`, `"a"` |

Đáp án được trả về **theo thứ tự bất kỳ** — nên khi tự kiểm tra, ta sẽ **sắp xếp** rồi mới so sánh.

### Constraints (ràng buộc)

- `1 <= words.length <= 5000`
- `0 <= words[i].length <= 300`
- `words[i]` chỉ gồm chữ cái tiếng Anh viết thường.

Để ý ngay:

- `n` (số từ) tới **5000** → số cặp có thứ tự là `n·(n-1) ≈ 2.5 * 10^7`. Thử hết các cặp, mỗi cặp lại kiểm tra một chuỗi dài tới 600 ký tự — rất nặng.
- `k` (độ dài một từ) chỉ tới **300** — nhỏ hơn `n` nhiều. Đây là gợi ý: nên làm việc **bên trong từng từ** (cỡ `k` hoặc `k²`) thay vì **giữa mọi cặp từ** (cỡ `n²`).
- Từ có thể là **chuỗi rỗng** `""` (độ dài 0) — Ví dụ 3 chính là ca này.

## 2. Hiểu đúng đề: cặp có thứ tự, `i != j`, chuỗi rỗng

Ba điều dễ hiểu sai:

1. **Cặp có thứ tự** (ordered pair): `[0, 1]` và `[1, 0]` là **hai cặp khác nhau**, vì `words[0] + words[1]` và `words[1] + words[0]` là hai chuỗi khác nhau. Ví dụ 1: `"s" + "lls" = "slls"` là palindrome, nhưng `"lls" + "s" = "llss"` thì **không**. Nên có `[3, 2]` mà không có `[2, 3]`.
2. **`i != j`**: không được ghép một từ với **chính nó**. Ví dụ `"aba" + "aba"` là palindrome nhưng không được tính, vì đó là cùng một chỉ số.
3. **Chuỗi rỗng** `""`: `"" + w = w` và `w + "" = w`. Vậy nếu `words` có `""` thì **mọi từ palindrome `w`** (khác `""`) cho ra **hai** cặp. Ví dụ 3: `"a"` là palindrome → `[0, 1]` và `[1, 0]`.

Vì `words` gồm các chuỗi **đôi một khác nhau** (unique), mỗi chuỗi ứng với **đúng một** chỉ số — điều này giúp ta dựng được bảng tra "từ → chỉ số" ở Cách 2.

Cell dưới kiểm tra trực tiếp các phép nối trong Ví dụ 1 để thấy thứ tự quan trọng thế nào.

In [1]:
words = ["abcd", "dcba", "lls", "s", "sssll"]

for i, j in [(0, 1), (1, 0), (3, 2), (2, 3), (2, 4), (4, 2)]:
    t = words[i] + words[j]
    print(f"[{i},{j}]  {words[i]!r:>8} + {words[j]!r:<8} = {t!r:<12} palindrome? {t == t[::-1]}")

print()
print("Ghép 'aba' với chính nó:", "aba" + "aba", "-> là palindrome nhưng i == j nên KHÔNG tính")
print("Chuỗi rỗng: '' + 'a' =", repr("" + "a"), " và  'a' + '' =", repr("a" + ""))

[0,1]    'abcd' + 'dcba'   = 'abcddcba'   palindrome? True
[1,0]    'dcba' + 'abcd'   = 'dcbaabcd'   palindrome? True
[3,2]       's' + 'lls'    = 'slls'       palindrome? True
[2,3]     'lls' + 's'      = 'llss'       palindrome? False
[2,4]     'lls' + 'sssll'  = 'llssssll'   palindrome? True
[4,2]   'sssll' + 'lls'    = 'ssslllls'   palindrome? False

Ghép 'aba' với chính nó: abaaba -> là palindrome nhưng i == j nên KHÔNG tính
Chuỗi rỗng: '' + 'a' = 'a'  và  'a' + '' = 'a'


## 3. Kiến thức nền 1 — Palindrome, cắt chuỗi (slicing) và đảo chuỗi

Với `w = "lls"` (độ dài `k = 3`), vị trí (index) đánh số từ 0: `w[0] = 'l'`, `w[1] = 'l'`, `w[2] = 's'`.

| Cú pháp | Đọc là | Với `w = "lls"` |
|---|---|---|
| `w[a:b]` | từ vị trí `a` đến **trước** `b` | `w[1:3] = "ls"` |
| `w[:j]` | **tiền tố** (prefix) — `j` ký tự đầu | `w[:2] = "ll"` |
| `w[j:]` | **hậu tố** (suffix) — phần còn lại từ vị trí `j` | `w[2:] = "s"` |
| `w[::-1]` | đảo ngược (bước nhảy `-1`) | `"sll"` |

Hai sự thật nhỏ nhưng **rất quan trọng** cho bài này:

- `w[:0] = ""` và `w[k:] = ""` — tiền tố/hậu tố có thể **rỗng**. Khi cho `j` chạy từ `0` đến `k` (tức `range(k + 1)`), ta có **`k + 1` cách cắt** `w = w[:j] + w[j:]`.
- **Chuỗi rỗng `""` là palindrome** (đảo ngược `""` vẫn là `""`), và **mọi chuỗi 1 ký tự** cũng là palindrome.

Cách kiểm tra palindrome đơn giản nhất trong Python: `t == t[::-1]`.

In [2]:
def is_palindrome(t: str) -> bool:
    """t đọc xuôi và đọc ngược có giống nhau không?"""
    return t == t[::-1]


w = "lls"
k = len(w)
print("w[1:3] =", repr(w[1:3]), "| w[:2] =", repr(w[:2]), "| w[2:] =", repr(w[2:]), "| w[::-1] =", repr(w[::-1]))
print("w[:0] =", repr(w[:0]), "  w[k:] =", repr(w[k:]))
print()
print(f"Có {k + 1} cách cắt '{w}' thành tiền tố | hậu tố:")
for j in range(k + 1):
    print(f"  j={j}:  {w[:j]!r:>6} | {w[j:]!r:<6}")
print()
for t in ["", "a", "ll", "lls", "abba", "abcd"]:
    print(f"is_palindrome({t!r:<6}) = {is_palindrome(t)}")

w[1:3] = 'ls' | w[:2] = 'll' | w[2:] = 's' | w[::-1] = 'sll'
w[:0] = ''   w[k:] = ''

Có 4 cách cắt 'lls' thành tiền tố | hậu tố:
  j=0:      '' | 'lls' 
  j=1:     'l' | 'ls'  
  j=2:    'll' | 's'   
  j=3:   'lls' | ''    

is_palindrome(''    ) = True
is_palindrome('a'   ) = True
is_palindrome('ll'  ) = True
is_palindrome('lls' ) = False
is_palindrome('abba') = True
is_palindrome('abcd') = False


## 4. Kiến thức nền 2 — Kiểm tra palindrome bằng hai con trỏ (two pointers), không cần cắt chuỗi

`t == t[::-1]` rất gọn, nhưng nó **tạo ra một chuỗi mới** (bản đảo ngược). Khi cần hỏi "đoạn `s[lo..hi]` có phải palindrome không?" hàng triệu lần, ta có thể kiểm tra **ngay trên chuỗi gốc** bằng **hai con trỏ** (two pointers):

- Con trỏ `lo` bắt đầu ở đầu đoạn, `hi` ở cuối đoạn.
- Nếu `s[lo] != s[hi]` → chắc chắn **không** phải palindrome, dừng ngay.
- Nếu bằng nhau → tiến `lo` sang phải, lùi `hi` sang trái, so cặp tiếp theo.
- Khi `lo >= hi` (hai con trỏ gặp nhau hoặc vượt nhau) → mọi cặp đã khớp → **là** palindrome.

Ví dụ `s = "sssll"`, hỏi đoạn `s[0..2] = "sss"`: so `s[0]` với `s[2]` ('s' = 's') → `lo = 1, hi = 1` → dừng, là palindrome.

Quy ước: đoạn **rỗng** ứng với `lo > hi` (ví dụ `lo = 5, hi = 4`) — vòng `while` không chạy lần nào và trả về `True`, đúng với "chuỗi rỗng là palindrome". Hàm này sẽ dùng ở Cách 3 (Trie).

In [3]:
def is_pal_range(s: str, lo: int, hi: int) -> bool:
    """Đoạn s[lo..hi] (tính cả hai đầu) có phải palindrome không? Đoạn rỗng (lo > hi) -> True."""
    while lo < hi:
        if s[lo] != s[hi]:
            return False
        lo += 1
        hi -= 1
    return True


s = "sssll"
print("s[0..2] =", repr(s[0:3]), "->", is_pal_range(s, 0, 2))
print("s[3..4] =", repr(s[3:5]), "->", is_pal_range(s, 3, 4))
print("s[1..4] =", repr(s[1:5]), "->", is_pal_range(s, 1, 4))
print("đoạn rỗng (lo=5, hi=4) ->", is_pal_range(s, 5, 4))

# Đối chiếu với cách cắt chuỗi trên MỌI đoạn của s (kể cả đoạn rỗng)
for lo in range(len(s) + 1):
    for hi in range(lo - 1, len(s)):
        assert is_pal_range(s, lo, hi) == is_palindrome(s[lo:hi + 1])
print("is_pal_range khớp is_palindrome trên mọi đoạn của", repr(s))

s[0..2] = 'sss' -> True
s[3..4] = 'll' -> True
s[1..4] = 'ssll' -> False
đoạn rỗng (lo=5, hi=4) -> True
is_pal_range khớp is_palindrome trên mọi đoạn của 'sssll'


## 5. Kiến thức nền 3 — So sánh hai đáp án "thứ tự bất kỳ"

Đề cho phép trả các cặp theo **thứ tự bất kỳ**. Lời giải của ta có thể ra `[[1,0],[0,1]]` trong khi đề in `[[0,1],[1,0]]` — **cả hai đều đúng**. Vậy không thể so bằng `==` trực tiếp.

Cách so chuẩn: **sắp xếp** cả hai rồi mới so.

- `sorted(list_of_lists)` sắp các list con theo **thứ tự từ điển** (lexicographic): so phần tử đầu, bằng thì so phần tử thứ hai... Ví dụ `sorted([[1,0],[0,1],[0,0]]) = [[0,0],[0,1],[1,0]]`.
- Muốn kiểm tra **không có cặp trùng**: đổi mỗi cặp sang `tuple` (list không bỏ vào `set` được vì list có thể thay đổi — không "hashable"), rồi so số phần tử của `set` với độ dài list.

Ta viết sẵn hàm `same_pairs(a, b)` và `has_duplicates(pairs)` để dùng xuyên suốt notebook.

In [4]:
def same_pairs(a, b) -> bool:
    """Hai danh sách cặp có giống nhau nếu bỏ qua thứ tự không?"""
    return sorted(a) == sorted(b)


def has_duplicates(pairs) -> bool:
    """Có cặp nào xuất hiện hơn một lần không?"""
    return len(set(map(tuple, pairs))) != len(pairs)


mine = [[1, 0], [0, 1]]
expected = [[0, 1], [1, 0]]
print("So bằng == :", mine == expected, "  <- sai lệch chỉ vì thứ tự")
print("So sau khi sắp xếp:", same_pairs(mine, expected))
print("sorted([[1,0],[0,1],[0,0]]) =", sorted([[1, 0], [0, 1], [0, 0]]))
print("has_duplicates([[0,1],[1,0],[0,1]]) =", has_duplicates([[0, 1], [1, 0], [0, 1]]))

So bằng == : False   <- sai lệch chỉ vì thứ tự
So sau khi sắp xếp: True
sorted([[1,0],[0,1],[0,0]]) = [[0, 0], [0, 1], [1, 0]]
has_duplicates([[0,1],[1,0],[0,1]]) = True


## 6. Cách 1 — Brute force: thử mọi cặp `(i, j)`

Cách người mới nghĩ ra đầu tiên chính là **làm đúng y như đề nói**:

1. Với mỗi `i` từ `0` đến `n - 1`,
2. với mỗi `j` từ `0` đến `n - 1` mà `j != i`,
3. nối `t = words[i] + words[j]`, nếu `t` là palindrome thì thêm `[i, j]` vào kết quả.

Hai vòng `for` lồng nhau duyệt **mọi cặp có thứ tự**. Ta dùng `enumerate(words)` để lấy cùng lúc **chỉ số** và **giá trị**: `for i, a in enumerate(["x", "y"])` cho `(0, "x")` rồi `(1, "y")`.

Cách này **chắc chắn đúng** (nó kiểm tra đúng định nghĩa), nên ta giữ nó làm **"oracle"** — bộ đối chiếu tin cậy — để kiểm tra các cách nhanh hơn ở phía sau.

In [5]:
def palindrome_pairs_brute(words):
    """Thử mọi cặp có thứ tự (i, j), i != j."""
    result = []
    for i, a in enumerate(words):
        for j, b in enumerate(words):
            if i != j and is_palindrome(a + b):
                result.append([i, j])
    return result


ex1 = ["abcd", "dcba", "lls", "s", "sssll"]
ex2 = ["bat", "tab", "cat"]
ex3 = ["a", ""]
for words, expected in [(ex1, [[0, 1], [1, 0], [3, 2], [2, 4]]),
                        (ex2, [[0, 1], [1, 0]]),
                        (ex3, [[0, 1], [1, 0]])]:
    got = palindrome_pairs_brute(words)
    assert same_pairs(got, expected)
    print(f"{words} -> {got}  PASS")

['abcd', 'dcba', 'lls', 's', 'sssll'] -> [[0, 1], [1, 0], [2, 4], [3, 2]]  PASS
['bat', 'tab', 'cat'] -> [[0, 1], [1, 0]]  PASS
['a', ''] -> [[0, 1], [1, 0]]  PASS


### Brute force chậm tới mức nào? Đo thử!

Với `n` từ, ta xét `n·(n-1)` cặp; mỗi cặp tốn `O(k)` để nối chuỗi và kiểm tra palindrome (chuỗi nối dài tới `2k`). Tổng: **`O(n²·k)`**.

Khi `n` tăng **gấp đôi**, số cặp tăng **gấp 4**. Cell dưới đo thời gian thật với `n = 250, 500, 1000` (mỗi từ dài 300 ký tự ngẫu nhiên), rồi ước lượng cho `n = 5000` bằng tỉ lệ `(5000 / 1000)² = 25`.

Công cụ mới: `time.perf_counter()` trả về mốc thời gian (giây) rất chính xác; lấy hiệu hai mốc là thời gian chạy. `random.seed(...)` cố định "hạt giống" để mỗi lần chạy lại sinh đúng dữ liệu cũ.

In [6]:
import random
import time


def random_words(n, length, alphabet="ab", seed=0):
    """Sinh n từ KHÁC NHAU, mỗi từ dài `length`, chữ cái lấy từ alphabet."""
    rng = random.Random(seed)
    seen = set()
    while len(seen) < n:
        seen.add("".join(rng.choice(alphabet) for _ in range(length)))
    return list(seen)


timings = {}
for n in [250, 500, 1000]:
    ws = random_words(n, 300, seed=n)
    start = time.perf_counter()
    palindrome_pairs_brute(ws)
    timings[n] = time.perf_counter() - start
    print(f"n={n:>4}: {n * (n - 1):>8} cặp, {timings[n]:.2f} giây")

print(f"Tỉ lệ thời gian n=1000 / n=500: khoảng {timings[1000] / timings[500]:.1f} lần (lý thuyết: 4)")
print(f"Ước lượng n=5000: khoảng {timings[1000] * 25:.0f} giây  -> quá chậm cho LeetCode")

n= 250:    62250 cặp, 0.03 giây


n= 500:   249500 cặp, 0.11 giây


n=1000:   999000 cặp, 0.48 giây
Tỉ lệ thời gian n=1000 / n=500: khoảng 4.5 lần (lý thuyết: 4)
Ước lượng n=5000: khoảng 12 giây  -> quá chậm cho LeetCode


### Nút thắt của Cách 1

Brute force chậm vì nó **ghép mù**: với mỗi từ `a`, nó thử **tất cả** `n - 1` từ `b` dù hầu hết chẳng có hy vọng. Với `a = "lls"`, hỏi "từ `b` nào làm `a + b` hoặc `b + a` thành palindrome?" — liệu ta có thể **suy ra luôn `b` phải trông như thế nào**, rồi chỉ việc **tra xem `b` đó có trong danh sách không**?

Đây đúng là tinh thần của bài Two Sum: thay vì thử mọi cặp, ta tính ra "phần còn thiếu" (`needed = target - x`) rồi tra trong hash map. Mục 7 sẽ tìm "phần còn thiếu" cho palindrome.

---

## 7. Quan sát then chốt — palindrome `A + B` trông như thế nào?

Giả sử `A + B` là palindrome. "Soi gương" chuỗi `A + B` quanh tâm của nó:

**Trường hợp 1: `len(A) >= len(B)`.** Đọc `A + B` từ cuối về đầu, `len(B)` ký tự cuối (chính là `B`) phải khớp với `len(B)` ký tự đầu của `A` đọc xuôi. Nghĩa là:

- `A` **bắt đầu bằng** `reverse(B)`, và
- phần còn lại của `A` (ở giữa) phải tự nó là palindrome.

```
A + B  =  [ reverse(B) | khúc giữa palindrome ] [ B ]
            \______________ A _______________/
```

Ví dụ cặp `[0, 1]` của Ví dụ 1: `A = "abcd"`, `B = "dcba"` → `A = reverse(B) + ""` (khúc giữa **rỗng** — vẫn là palindrome). Một ví dụ có khúc giữa không rỗng: `A = "abxyx"`, `B = "ba"` → `A = "ab" + "xyx"`, và `A + B = "abxyxba"` ✔.

**Trường hợp 2: `len(A) < len(B)`.** Đối xứng lại:

- `B` **kết thúc bằng** `reverse(A)`, và
- phần đầu còn lại của `B` phải là palindrome.

```
A + B  =  [ A ] [ khúc giữa palindrome | reverse(A) ]
                  \_______________ B _____________/
```

Ví dụ cặp `[3, 2]`: `A = "s"`, `B = "lls"` → `B = "ll" + "s"`, khúc `"ll"` là palindrome, `"s" = reverse(A)` ✔. Cặp `[2, 4]`: `A = "lls"`, `B = "sssll"` → `B = "ss" + "sll"`, `"ss"` palindrome, `"sll" = reverse("lls")` ✔.

**Chiều ngược lại cũng đúng**: nếu `A = reverse(B) + P` với `P` palindrome thì `A + B = reverse(B) + P + B` — soi gương thấy ngay là palindrome. Tương tự cho trường hợp 2. Vậy đây là **điều kiện cần và đủ**.

Cell dưới kiểm chứng cách "bóc tách" này trên mọi cặp của Ví dụ 1.

In [7]:
def explain_pair(A: str, B: str) -> str:
    """Bóc tách palindrome A + B theo quan sát ở mục 7."""
    if len(A) >= len(B):
        head, mid = A[:len(B)], A[len(B):]
        assert head == B[::-1] and is_palindrome(mid)
        return f"TH1: A = reverse(B) {head!r} + khúc giữa palindrome {mid!r}"
    else:
        mid, tail = B[:len(B) - len(A)], B[len(B) - len(A):]
        assert tail == A[::-1] and is_palindrome(mid)
        return f"TH2: B = khúc giữa palindrome {mid!r} + reverse(A) {tail!r}"


words = ["abcd", "dcba", "lls", "s", "sssll"]
for i, j in palindrome_pairs_brute(words):
    A, B = words[i], words[j]
    print(f"[{i},{j}] A={A!r:<8} B={B!r:<8} -> {explain_pair(A, B)}")

[0,1] A='abcd'   B='dcba'   -> TH1: A = reverse(B) 'abcd' + khúc giữa palindrome ''
[1,0] A='dcba'   B='abcd'   -> TH1: A = reverse(B) 'dcba' + khúc giữa palindrome ''
[2,4] A='lls'    B='sssll'  -> TH2: B = khúc giữa palindrome 'ss' + reverse(A) 'sll'
[3,2] A='s'      B='lls'    -> TH2: B = khúc giữa palindrome 'll' + reverse(A) 's'


## 8. Đổi góc nhìn: đứng ở **một từ** `w` và cắt nó làm đôi

Quan sát mục 7 nói: trong mỗi cặp palindrome, **từ dài hơn (hoặc bằng)** chứa *một khúc palindrome* + *đảo ngược của từ kia*. Vậy nếu ta đứng ở từ `w` và **cắt `w` tại mọi vị trí `j`** thành `prefix = w[:j]` và `suffix = w[j:]`, sẽ có hai luật:

**Luật A — `prefix` là palindrome** (khúc palindrome nằm ở **đầu** `w`):
Nếu `reverse(suffix)` là một từ `x` có trong danh sách, thì `x + w` là palindrome:
`x + w = reverse(suffix) + prefix + suffix` → khúc `prefix` palindrome ở giữa, hai bên là gương của nhau. → thêm cặp **`[chỉ số của x, chỉ số của w]`** (`x` đứng **trước**).

**Luật B — `suffix` là palindrome** (khúc palindrome nằm ở **cuối** `w`):
Nếu `reverse(prefix)` là một từ `x` có trong danh sách, thì `w + x` là palindrome:
`w + x = prefix + suffix + reverse(prefix)` → thêm cặp **`[chỉ số của w, chỉ số của x]`** (`w` đứng **trước**).

Ví dụ `w = "lls"`:

| `j` | `prefix` | `suffix` | prefix pal? | Luật A: tìm `reverse(suffix)` | suffix pal? | Luật B: tìm `reverse(prefix)` |
|:-:|---|---|:-:|---|:-:|---|
| 0 | `""` | `"lls"` | ✔ | `"sll"` | ✘ | — |
| 1 | `"l"` | `"ls"` | ✔ | `"sl"` | ✘ | — |
| 2 | `"ll"` | `"s"` | ✔ | **`"s"`** ← có trong Ví dụ 1! → `"s" + "lls"` | ✔ | `"ll"` |
| 3 | `"lls"` | `""` | ✘ | — | ✔ | `"sll"` |

Thay vì thử `n` từ, mỗi từ `w` chỉ cần **`k + 1` lát cắt** × **2 lần tra cứu**. Để "tra cứu xem một chuỗi có trong danh sách không, và ở chỉ số nào" thật nhanh, ta cần **hash map** — mục 9.

In [8]:
w = "lls"
print(f"{'j':>2} | {'prefix':<7} | {'suffix':<7} | pre pal? | A tìm  | suf pal? | B tìm")
for j in range(len(w) + 1):
    prefix, suffix = w[:j], w[j:]
    a_need = repr(suffix[::-1]) if is_palindrome(prefix) else "-"
    b_need = repr(prefix[::-1]) if is_palindrome(suffix) else "-"
    print(f"{j:>2} | {prefix!r:<7} | {suffix!r:<7} | {is_palindrome(prefix)!s:<8} | {a_need:<6} | "
          f"{is_palindrome(suffix)!s:<8} | {b_need}")

 j | prefix  | suffix  | pre pal? | A tìm  | suf pal? | B tìm
 0 | ''      | 'lls'   | True     | 'sll'  | False    | -
 1 | 'l'     | 'ls'    | True     | 'sl'   | False    | -
 2 | 'll'    | 's'     | True     | 's'    | True     | 'll'
 3 | 'lls'   | ''      | False    | -      | True     | 'sll'


## 9. Kiến thức nền 4 — Hash map (`dict`): tra "từ → chỉ số" thật nhanh

Một **dictionary** (`dict`, còn gọi là **hash map** — bảng băm) lưu các cặp **khóa → giá trị** (key → value). Ở bài này: **khóa là từ**, **giá trị là chỉ số** của từ đó trong `words`.

- Tạo nhanh bằng **dict comprehension**: `{w: i for i, w in enumerate(words)}` — "với mỗi `(i, w)`, lưu `w → i`".
- `x in index` hỏi "chuỗi `x` có phải một khóa không?" — **trung bình `O(1)` lần tra**, cộng thêm chi phí tính **hash** (mã băm) của `x`, tỉ lệ với độ dài `x`, tức `O(len(x))`.
- `index[x]` đọc ra chỉ số đã lưu; `index.get(x, -1)` trả `-1` nếu không có (không gây lỗi `KeyError`).

So sánh với cách "ngây thơ" là `words.index(x)`: nó phải **duyệt cả list** từ đầu tới khi gặp → `O(n)` lần so sánh mỗi lần tra, và còn **báo lỗi** `ValueError` nếu không có. Với `n = 5000` và hàng triệu lần tra, khác biệt này là rất lớn.

Vì đề cho các từ **đôi một khác nhau**, mỗi khóa ứng với đúng một chỉ số — không lo bị ghi đè.

In [9]:
words = ["abcd", "dcba", "lls", "s", "sssll"]
index = {w: i for i, w in enumerate(words)}
print("index =", index)
print("'s' in index      :", "s" in index, "-> chỉ số", index["s"])
print("'sll' in index    :", "sll" in index)
print("index.get('sll', -1) =", index.get("sll", -1))
print("'' in index (Ví dụ 1 không có chuỗi rỗng):", "" in index)

index3 = {w: i for i, w in enumerate(["a", ""])}
print("Ví dụ 3:", index3, "-> chuỗi rỗng cũng là một khóa hợp lệ, chỉ số", index3[""])

try:
    words.index("sll")
except ValueError as e:
    print("words.index('sll') báo lỗi:", e)

index = {'abcd': 0, 'dcba': 1, 'lls': 2, 's': 3, 'sssll': 4}
's' in index      : True -> chỉ số 3
'sll' in index    : False
index.get('sll', -1) = -1
'' in index (Ví dụ 1 không có chuỗi rỗng): False
Ví dụ 3: {'a': 0, '': 1} -> chuỗi rỗng cũng là một khóa hợp lệ, chỉ số 1
words.index('sll') báo lỗi: 'sll' is not in list


## 10. Cách 2 — Ghép hai luật với hash map (phiên bản đầu tiên, còn một lỗi)

Ráp mọi thứ lại:

1. Dựng `index = {từ: chỉ số}`.
2. Với mỗi từ `w` (chỉ số `i`), với mỗi lát cắt `j` từ `0` đến `k` (`k = len(w)`):
   - **Luật A**: nếu `prefix = w[:j]` là palindrome và `x = reverse(suffix)` có trong `index` → thêm `[index[x], i]`.
   - **Luật B**: nếu `suffix = w[j:]` là palindrome và `x = reverse(prefix)` có trong `index` → thêm `[i, index[x]]`.
3. **Không ghép với chính mình**: chỉ thêm khi `index[x] != i`. Ví dụ `w = "aba"`, lát cắt `j = 0`: `prefix = ""` palindrome, `reverse("aba") = "aba"` — chính là `w`! Nếu không chặn, ta sẽ thêm `[i, i]`.

Cell dưới là bản **ngây thơ** đúng y các bước trên. Chạy thử trên Ví dụ 1 — để ý số cặp trả về.

In [10]:
def palindrome_pairs_hashmap_naive(words):
    """Bản đầu tiên: đúng ý tưởng nhưng CHƯA chặn trùng lặp."""
    index = {w: i for i, w in enumerate(words)}
    result = []
    for i, w in enumerate(words):
        for j in range(len(w) + 1):
            prefix, suffix = w[:j], w[j:]
            if is_palindrome(prefix):                       # Luật A
                x = suffix[::-1]
                if x in index and index[x] != i:
                    result.append([index[x], i])
            if is_palindrome(suffix):                       # Luật B
                x = prefix[::-1]
                if x in index and index[x] != i:
                    result.append([i, index[x]])
    return result


got = palindrome_pairs_hashmap_naive(["abcd", "dcba", "lls", "s", "sssll"])
print("Kết quả:", got)
print("Số cặp:", len(got), "(đề chỉ có 4)  | có trùng?", has_duplicates(got))
print("Ví dụ 3:", palindrome_pairs_hashmap_naive(["a", ""]))

Kết quả: [[1, 0], [0, 1], [0, 1], [1, 0], [3, 2], [2, 4]]
Số cặp: 6 (đề chỉ có 4)  | có trùng? True
Ví dụ 3: [[0, 1], [1, 0]]


## 11. Vì sao bị trùng, và sửa bằng một điều kiện nhỏ

Cặp `[1, 0]` (`"dcba" + "abcd"`) bị sinh ra **hai lần**:

- Từ `w = "abcd"` (i = 0), lát cắt `j = 0`: `prefix = ""` palindrome, `reverse("abcd") = "dcba"` có → Luật A thêm `[1, 0]`.
- Từ `w = "dcba"` (i = 1), lát cắt `j = 4 = k`: `suffix = ""` palindrome, `reverse("dcba") = "abcd"` có → Luật B thêm `[1, 0]`.

Cả hai lần đều là trường hợp **"từ kia đúng bằng đảo ngược của *toàn bộ* `w`"** — tức hai từ **dài bằng nhau**. Luật A ở `j = 0` và Luật B ở `j = k` cùng bắt một loại cặp, chỉ là đứng ở hai từ khác nhau.

**Cách sửa:** chỉ cho Luật B chạy khi **`j < k`** (bỏ lát cắt có `suffix` rỗng ở Luật B). Cặp hai từ dài bằng nhau vẫn được Luật A bắt ở `j = 0` của từ đứng **sau** — không mất gì.

Liệu còn chỗ nào trùng hay sót nữa không? Hãy phân loại cặp đúng `[a, b]` (từ `A = words[a]` đứng trước) theo độ dài — đây chính là hai trường hợp của mục 7:

| Loại cặp | Ai bắt được? | Lát cắt |
|---|---|---|
| `len(A) > len(B)` | Luật B, đứng ở `A` | `j = len(B)`, và `j < len(A)` ✔ |
| `len(A) == len(B)` | Luật A, đứng ở `B` | `j = 0` |
| `len(A) < len(B)` | Luật A, đứng ở `B` | `j = len(B) - len(A) > 0` |

Mỗi loại có **đúng một** "nơi" bắt nó (mục 21 sẽ chứng minh kỹ) → không sót, không trùng.

Một hệ quả dễ chịu: **chuỗi rỗng không cần xử lý riêng**. Với `w = "a"` trong Ví dụ 3: Luật B ở `j = 0` (`suffix = "a"`, tìm `""`) cho `[0, 1]`; Luật A ở `j = 1` (`prefix = "a"`, tìm `reverse("") = ""`) cho `[1, 0]`.

In [11]:
def palindrome_pairs_hashmap(words):
    """Cách 2 — hash map + cắt từng từ. Thời gian O(n·k²), bộ nhớ O(tổng độ dài) ngoài kết quả."""
    index = {w: i for i, w in enumerate(words)}
    result = []
    for i, w in enumerate(words):
        k = len(w)
        for j in range(k + 1):                              # k + 1 lát cắt, kể cả 2 lát có phần rỗng
            prefix, suffix = w[:j], w[j:]
            # Luật A: prefix palindrome -> x = reverse(suffix) đứng TRƯỚC w
            if is_palindrome(prefix):
                x = suffix[::-1]
                if x in index and index[x] != i:
                    result.append([index[x], i])
            # Luật B: suffix palindrome -> x = reverse(prefix) đứng SAU w   (j < k để tránh trùng)
            if j < k and is_palindrome(suffix):
                x = prefix[::-1]
                if x in index and index[x] != i:
                    result.append([i, index[x]])
    return result


for words, expected in [(ex1, [[0, 1], [1, 0], [3, 2], [2, 4]]),
                        (ex2, [[0, 1], [1, 0]]),
                        (ex3, [[0, 1], [1, 0]])]:
    got = palindrome_pairs_hashmap(words)
    assert same_pairs(got, expected) and not has_duplicates(got)
    print(f"{words} -> {got}  PASS")

['abcd', 'dcba', 'lls', 's', 'sssll'] -> [[1, 0], [0, 1], [3, 2], [2, 4]]  PASS
['bat', 'tab', 'cat'] -> [[1, 0], [0, 1]]  PASS
['a', ''] -> [[0, 1], [1, 0]]  PASS


## 12. Trace Cách 2 từng lát cắt

Hàm `trace_hashmap` in **mọi** lát cắt của mọi từ, mỗi dòng gồm:

- `w[i]` và lát cắt `j`, dạng `prefix | suffix`;
- **A:** nếu `prefix` palindrome — chuỗi cần tìm `reverse(suffix)`, có hay không, và cặp được thêm;
- **B:** nếu `suffix` palindrome và `j < k` — chuỗi cần tìm `reverse(prefix)`; nếu `j = k` thì ghi rõ "bỏ qua";
- Luật nào không áp dụng (vì khúc tương ứng không phải palindrome) thì không in.

Chạy trên Ví dụ 1 trước.

In [12]:
def trace_hashmap(words):
    index = {w: i for i, w in enumerate(words)}
    result = []
    for i, w in enumerate(words):
        k = len(w)
        for j in range(k + 1):
            prefix, suffix = w[:j], w[j:]
            notes = []
            if is_palindrome(prefix):
                x = suffix[::-1]
                if x not in index:
                    notes.append(f"A: tìm {x!r} -> không có")
                elif index[x] == i:
                    notes.append(f"A: tìm {x!r} -> chính nó, bỏ")
                else:
                    result.append([index[x], i])
                    notes.append(f"A: tìm {x!r} -> có [{index[x]}] => THÊM {[index[x], i]}")
            if is_palindrome(suffix):
                x = prefix[::-1]
                if j == k:
                    notes.append("B: j = k, bỏ qua")
                elif x not in index:
                    notes.append(f"B: tìm {x!r} -> không có")
                elif index[x] == i:
                    notes.append(f"B: tìm {x!r} -> chính nó, bỏ")
                else:
                    result.append([i, index[x]])
                    notes.append(f"B: tìm {x!r} -> có [{index[x]}] => THÊM {[i, index[x]]}")
            cut = f"{prefix}|{suffix}"
            print(f"w[{i}]={w!r:<8} j={j}  {cut:<8} " + ("; ".join(notes) if notes else "(không luật nào)"))
    print("Kết quả:", result)
    return result


trace_hashmap(["abcd", "dcba", "lls", "s", "sssll"]);

w[0]='abcd'   j=0  |abcd    A: tìm 'dcba' -> có [1] => THÊM [1, 0]
w[0]='abcd'   j=1  a|bcd    A: tìm 'dcb' -> không có
w[0]='abcd'   j=2  ab|cd    (không luật nào)
w[0]='abcd'   j=3  abc|d    B: tìm 'cba' -> không có
w[0]='abcd'   j=4  abcd|    B: j = k, bỏ qua
w[1]='dcba'   j=0  |dcba    A: tìm 'abcd' -> có [0] => THÊM [0, 1]
w[1]='dcba'   j=1  d|cba    A: tìm 'abc' -> không có
w[1]='dcba'   j=2  dc|ba    (không luật nào)
w[1]='dcba'   j=3  dcb|a    B: tìm 'bcd' -> không có
w[1]='dcba'   j=4  dcba|    B: j = k, bỏ qua
w[2]='lls'    j=0  |lls     A: tìm 'sll' -> không có
w[2]='lls'    j=1  l|ls     A: tìm 'sl' -> không có
w[2]='lls'    j=2  ll|s     A: tìm 's' -> có [3] => THÊM [3, 2]; B: tìm 'll' -> không có
w[2]='lls'    j=3  lls|     B: j = k, bỏ qua
w[3]='s'      j=0  |s       A: tìm 's' -> chính nó, bỏ; B: tìm '' -> không có
w[3]='s'      j=1  s|       A: tìm '' -> không có; B: j = k, bỏ qua
w[4]='sssll'  j=0  |sssll   A: tìm 'llsss' -> không có
w[4]='sssll'  j=1  s|ssll   A: tìm

### Đọc kết quả trace (Ví dụ 1)

Chỉ có 4 dòng "THÊM", khớp đúng 4 cặp của đề:

| Từ đang đứng | `j` | `prefix` | `suffix` | Luật | Tìm | Cặp thêm | Palindrome |
|---|:-:|---|---|:-:|---|---|---|
| `w[0] = "abcd"` | 0 | `""` | `"abcd"` | A | `"dcba"` → [1] | `[1, 0]` | `"dcbaabcd"` |
| `w[1] = "dcba"` | 0 | `""` | `"dcba"` | A | `"abcd"` → [0] | `[0, 1]` | `"abcddcba"` |
| `w[2] = "lls"` | 2 | `"ll"` | `"s"` | A | `"s"` → [3] | `[3, 2]` | `"slls"` |
| `w[4] = "sssll"` | 2 | `"ss"` | `"sll"` | A | `"lls"` → [2] | `[2, 4]` | `"llssssll"` |

Vài dòng đáng chú ý:

- `w[0]`, `j = 4` (`prefix = "abcd"`, `suffix = ""`): `suffix` rỗng là palindrome nhưng Luật B **bỏ qua** vì `j = k`. Nếu không bỏ, nó sẽ thêm lại `[0, 1]` — trùng với dòng của `w[1]`, `j = 0`.
- `w[3] = "s"`, `j = 0`: Luật A tìm `reverse("s") = "s"` — **chính nó**, bỏ (chặn `i == j`).
- Các dòng `ab|cd` và `dc|ba` (`j = 2`): cả `prefix` lẫn `suffix` đều không phải palindrome → không cần tra gì cả. Đây là chỗ Cách 2 tiết kiệm so với brute force.
- Để ý: cả 4 cặp đều do Luật A bắt — vì trong Ví dụ 1, từ đứng **sau** luôn dài hơn hoặc bằng từ đứng trước. Ví dụ tiếp theo sẽ có Luật B.

---

### Trace thứ hai: chuỗi rỗng + ghép với chính mình + Luật B

`words = ["aba", "", "ab"]` có đủ các nhánh đặc biệt. Đoán trước bằng brute force: `"aba" + ""`, `"" + "aba"` và `"ab" + "aba" = "ababa"` là palindrome → đáp án `[[0,1],[1,0],[2,0]]`.

Hãy để ý trong trace:

- `w[0] = "aba"`, `j = 0`: Luật A tìm `"aba"` → chính nó, bỏ; Luật B (`suffix = "aba"` palindrome) tìm `reverse("") = ""` → có ở chỉ số 1 → `[0, 1]`.
- `w[0]`, `j = 1` (`prefix = "a"`, `suffix = "ba"`): `prefix = "a"` palindrome, tìm `reverse("ba") = "ab"` → `[2, 0]`, tức `"ab" + "aba"`.
- `w[0]`, `j = 3` (`prefix = "aba"`, `suffix = ""`): Luật A tìm `reverse("") = ""` → `[1, 0]`.
- `w[1] = ""` chỉ có **một** lát cắt (`j = 0 = k`): Luật A tìm `""` → chính nó; Luật B bỏ qua vì `j = k`.

In [13]:
got = trace_hashmap(["aba", "", "ab"])
assert same_pairs(got, palindrome_pairs_brute(["aba", "", "ab"]))

w[0]='aba'    j=0  |aba     A: tìm 'aba' -> chính nó, bỏ; B: tìm '' -> có [1] => THÊM [0, 1]
w[0]='aba'    j=1  a|ba     A: tìm 'ab' -> có [2] => THÊM [2, 0]
w[0]='aba'    j=2  ab|a     B: tìm 'ba' -> không có
w[0]='aba'    j=3  aba|     A: tìm '' -> có [1] => THÊM [1, 0]; B: j = k, bỏ qua
w[1]=''       j=0  |        A: tìm '' -> chính nó, bỏ; B: j = k, bỏ qua
w[2]='ab'     j=0  |ab      A: tìm 'ba' -> không có
w[2]='ab'     j=1  a|b      A: tìm 'b' -> không có; B: tìm 'a' -> không có
w[2]='ab'     j=2  ab|      B: j = k, bỏ qua
Kết quả: [[0, 1], [2, 0], [1, 0]]


## 13. Độ phức tạp của Cách 2 — và yêu cầu "`O(tổng độ dài)`" của đề

Ký hiệu: `n` = số từ, `k` = độ dài từ dài nhất, `S` = tổng độ dài mọi từ (`S <= n·k`).

- **Dựng `index`**: băm mỗi từ một lần → `O(S)`.
- **Mỗi từ** có `k + 1` lát cắt. Mỗi lát cắt tốn `O(k)`: cắt chuỗi, đảo chuỗi, kiểm tra palindrome, tính hash để tra `index`.
- **Tổng thời gian: `O(n·k²)`** (chính xác hơn: tổng của `len(w)²` trên mọi từ). Với `n = 5000`, `k = 300`: cỡ `5000 · 300 · 300 = 4.5 * 10^8` thao tác **trên từng ký tự** — nghe lớn, nhưng các thao tác này (slicing, so sánh, hash) chạy bằng mã C bên trong Python nên thực tế chỉ mất khoảng vài giây hoặc ít hơn.
- **Bộ nhớ**: `O(S)` cho `index` + các chuỗi tạm `O(k)`, chưa kể mảng kết quả.

So với brute force `O(n²·k)` (cỡ `2.5 * 10^7 · 600`), ta đã đổi **`n²` thành `n·k`** — đúng như gợi ý từ constraints: `k = 300` nhỏ hơn `n = 5000` nhiều.

**Về yêu cầu `O(tổng độ dài)` (`O(S)`) của đề:** Cách 2 là `O(n·k²)`, chưa đạt `O(S)` về lý thuyết. Tuy vậy đây là lời giải phổ biến nhất và **được chấp nhận** trên LeetCode (đề không có cách nào "đo" Big-O; chỉ có giới hạn thời gian). Cách 3 (Trie) là hướng được nhắc nhiều nhất cho bài này nhưng vẫn `O(n·k²)` vì còn kiểm tra palindrome; Cách 4 (mục 16–18) mới thật sự loại được thừa số `k` cuối cùng.

Cell dưới đo Cách 2 ở cỡ tối đa của constraints với hai kiểu dữ liệu: từ ngẫu nhiên, và từ **toàn chữ `'a'`** (`"", "a", "aa", ...`) — kiểu này mọi lát cắt đều là palindrome và **mọi** cặp đều hợp lệ (`301 · 300 = 90300` cặp).

In [14]:
ws_random = random_words(5000, 300, seed=336)
start = time.perf_counter()
r = palindrome_pairs_hashmap(ws_random)
print(f"5000 từ ngẫu nhiên dài 300: {len(r)} cặp, {time.perf_counter() - start:.2f} giây")

ws_all_a = ["a" * L for L in range(301)]            # 301 từ khác nhau: '', 'a', 'aa', ..., 'a'*300
start = time.perf_counter()
r = palindrome_pairs_hashmap(ws_all_a)
print(f"301 từ toàn 'a':           {len(r)} cặp, {time.perf_counter() - start:.2f} giây")
assert len(r) == 301 * 300 and not has_duplicates(r)

5000 từ ngẫu nhiên dài 300: 0 cặp, 0.66 giây
301 từ toàn 'a':           90300 cặp, 0.06 giây


## 14. Kiến thức nền 5 — Trie (cây tiền tố) từ con số 0

**Trie** (đọc là "try", còn gọi là *prefix tree* — cây tiền tố) là một cây để lưu nhiều chuỗi sao cho các chuỗi **có chung tiền tố thì dùng chung đường đi**.

- Mỗi **nút** (node) có một `dict` tên `children`: ký tự → nút con.
- Đi từ **gốc** (root) xuống theo các ký tự của chuỗi `"car"`: gốc →`'c'`→`'a'`→`'r'`. Nút cuối được **đánh dấu** "có một từ kết thúc ở đây".
- Thêm `"cart"`: đi lại đường `c → a → r` có sẵn, chỉ tạo thêm nút `'t'`. Thêm `"cat"`: dùng chung `c → a`, rẽ nhánh `'t'`.

```
(gốc)
 ├─ c ─ a ─┬─ r* ─ t*        * = có từ kết thúc ở nút này
 │         └─ t*
 └─ d ─ o ─ g*
```

Ba thao tác cơ bản, mỗi thao tác đi xuống tối đa `len(chuỗi)` bước → `O(len)`:

| Thao tác | Ý nghĩa | Cách làm |
|---|---|---|
| `insert(w)` | thêm từ | đi xuống, thiếu nút thì tạo; đánh dấu nút cuối |
| `search(w)` | `w` có phải một từ đã thêm? | đi xuống; hỏng giữa đường → `False`; tới nơi thì xem dấu |
| `starts_with(p)` | có từ nào bắt đầu bằng `p`? | đi xuống hết `p` được là `True` |

Công cụ Python: `node.children.setdefault(ch, TrieNode())` — "nếu chưa có con `ch` thì tạo nút mới, rồi trả về con `ch`". `__slots__` khai báo trước các thuộc tính của class, giúp mỗi nút tốn **ít bộ nhớ hơn** (quan trọng khi có hàng trăm nghìn nút).

In [15]:
class SimpleTrieNode:
    __slots__ = ("children", "is_end")

    def __init__(self):
        self.children = {}      # ký tự -> SimpleTrieNode
        self.is_end = False     # có từ nào kết thúc ở nút này?


class SimpleTrie:
    def __init__(self):
        self.root = SimpleTrieNode()

    def insert(self, w):
        node = self.root
        for ch in w:
            node = node.children.setdefault(ch, SimpleTrieNode())
        node.is_end = True

    def _walk(self, s):
        node = self.root
        for ch in s:
            node = node.children.get(ch)
            if node is None:
                return None                  # hỏng giữa đường
        return node

    def search(self, w):
        node = self._walk(w)
        return node is not None and node.is_end

    def starts_with(self, p):
        return self._walk(p) is not None


def print_simple_trie(node, depth=0):
    for ch, child in sorted(node.children.items()):
        print("    " * depth + ch + ("*" if child.is_end else ""))
        print_simple_trie(child, depth + 1)


t = SimpleTrie()
for w in ["car", "cart", "cat", "dog"]:
    t.insert(w)
print_simple_trie(t.root)
print("search('car') =", t.search("car"), "| search('ca') =", t.search("ca"),
      "| starts_with('ca') =", t.starts_with("ca"), "| starts_with('cb') =", t.starts_with("cb"))

c
    a
        r*
            t*
        t*
d
    o
        g*
search('car') = True | search('ca') = False | starts_with('ca') = True | starts_with('cb') = False


## 15. Cách 3 — Trie trên các **từ đảo ngược**

Cách 2 tốn công ở chỗ: mỗi lát cắt lại **tạo chuỗi mới** `reverse(suffix)` / `reverse(prefix)` rồi băm nó để tra `dict`. Trie cho phép "tra dần từng ký tự": đi xuống thêm một bước là biết ngay tiền tố dài thêm 1 còn khớp hay không, không cần băm lại cả chuỗi.

### Ý tưởng

Bỏ **đảo ngược** của mọi từ vào Trie. Rồi với mỗi từ `w` (chỉ số `i`), **đi xuống Trie theo từng ký tự của `w`** và tìm mọi `x` sao cho `w + x` là palindrome (`w` đứng **trước**). Khi đang ở độ sâu `j` (đã đi hết `w[:j]`), mỗi từ đảo ngược nằm dưới nút này **bắt đầu bằng `w[:j]`**. Có hai tình huống:

**Tình huống 1 — `x` ngắn hơn `w`** (một từ đảo ngược **kết thúc giữa đường**, ở độ sâu `j < k`):
`reverse(x) = w[:j]` → `x = reverse(w[:j])`. Khi đó `w + x = w[:j] + w[j:] + reverse(w[:j])` là palindrome ⟺ **`w[j:]` là palindrome**. (Đây đúng là **Luật B** của Cách 2!)

**Tình huống 2 — `x` dài hơn hoặc bằng `w`** (đi **hết** `w`, các từ đảo ngược nằm **bên dưới** nút cuối):
`reverse(x) = w + rest` → `x = reverse(rest) + reverse(w)`. Khi đó `w + x = w + reverse(rest) + reverse(w)` là palindrome ⟺ **`rest` là palindrome**.

Để tình huống 2 nhanh, lúc thêm từ vào Trie ta **chuẩn bị sẵn**: ở mỗi nút, lưu danh sách `pal_below` = chỉ số các từ đi **qua** nút này mà **phần còn lại bên dưới là palindrome**. Đi hết `w` là có ngay danh sách đáp án, khỏi duyệt cây con.

### Mỗi nút lưu gì?

| Thuộc tính | Ý nghĩa |
|---|---|
| `children` | ký tự → nút con |
| `word_id` | chỉ số của từ mà **đảo ngược** của nó kết thúc đúng ở nút này (`-1` nếu không có) |
| `pal_below` | chỉ số các từ `x` có `reverse(x)` đi qua nút này và **phần còn lại** (kể cả rỗng) là palindrome |

### Thêm từ `x` (chỉ số `e`) vào Trie

Đặt `r = reverse(x)`. Đi xuống theo `r`; **trước khi** đi theo ký tự `r[d]` (đang ở độ sâu `d`), nếu `r[d:]` là palindrome thì thêm `e` vào `pal_below` của nút hiện tại. Tới nút cuối: gán `word_id = e` và thêm `e` vào `pal_below` (phần còn lại rỗng — cũng là palindrome).

Cell dưới dựng Trie cho Ví dụ 1 và in **mọi nút** (kèm đường đi từ gốc) để bạn tự soi.

In [16]:
class TrieNode:
    __slots__ = ("children", "word_id", "pal_below")

    def __init__(self):
        self.children = {}
        self.word_id = -1          # từ có đảo ngược kết thúc ở đây
        self.pal_below = []        # các từ có "phần còn lại bên dưới" là palindrome


def build_reversed_trie(words):
    root = TrieNode()
    for e, x in enumerate(words):
        r = x[::-1]
        node = root
        for d, ch in enumerate(r):
            if is_pal_range(r, d, len(r) - 1):       # phần còn lại r[d:] là palindrome?
                node.pal_below.append(e)
            node = node.children.setdefault(ch, TrieNode())
        node.word_id = e
        node.pal_below.append(e)                     # phần còn lại rỗng
    return root


def print_trie(node, path=""):
    label = path if path else "(gốc)"
    wid = node.word_id if node.word_id != -1 else "-"
    print(f"{label:<7} word_id={wid!s:<2} pal_below={node.pal_below}")
    for ch, child in sorted(node.children.items()):
        print_trie(child, path + ch)


words = ["abcd", "dcba", "lls", "s", "sssll"]
print("Các từ đảo ngược:", [w[::-1] for w in words])
print_trie(build_reversed_trie(words))

Các từ đảo ngược: ['dcba', 'abcd', 'sll', 's', 'llsss']
(gốc)   word_id=-  pal_below=[3]
a       word_id=-  pal_below=[]
ab      word_id=-  pal_below=[]
abc     word_id=-  pal_below=[1]
abcd    word_id=1  pal_below=[1]
d       word_id=-  pal_below=[]
dc      word_id=-  pal_below=[]
dcb     word_id=-  pal_below=[0]
dcba    word_id=0  pal_below=[0]
l       word_id=-  pal_below=[]
ll      word_id=-  pal_below=[4]
lls     word_id=-  pal_below=[4]
llss    word_id=-  pal_below=[4]
llsss   word_id=4  pal_below=[4]
s       word_id=3  pal_below=[2, 3]
sl      word_id=-  pal_below=[2]
sll     word_id=2  pal_below=[2]


### Soi vài nút trong Trie của Ví dụ 1

- Nút `s` (đường đi `"s"`): `word_id = 3` vì `reverse("s") = "s"` kết thúc ở đây. `pal_below = [2, 3]`: từ 3 (phần còn lại rỗng) và từ 2 — vì `reverse("lls") = "sll"` đi qua `s` và phần còn lại `"ll"` là palindrome.
- Nút `ll` (đường đi `"ll"`): `pal_below = [4]` vì `reverse("sssll") = "llsss"` đi qua `"ll"`, phần còn lại `"sss"` là palindrome.
- Gốc: `pal_below = [3]` — chỉ từ `"s"` là có **toàn bộ** đảo ngược là palindrome (tức bản thân từ là palindrome).

### Tìm cặp cho từ `w` (chỉ số `i`)

```
node = gốc
for j, ch in enumerate(w):
    # Tình huống 1: có từ x kết thúc ở đây (x ngắn hơn w)
    if node.word_id != -1 and node.word_id != i and w[j:] là palindrome:
        thêm [i, node.word_id]
    node = node.children.get(ch);   nếu None thì dừng (không còn từ nào khớp)
nếu đi hết w (không dừng giữa chừng):
    # Tình huống 2: x dài hơn hoặc bằng w
    với mỗi e trong node.pal_below mà e != i: thêm [i, e]
```

Python có cú pháp `for ... else`: khối `else` **chỉ chạy khi vòng `for` kết thúc bình thường** (không bị `break`). Rất hợp để viết "nếu đi hết `w`".

In [17]:
def palindrome_pairs_trie(words):
    """Cách 3 — Trie trên các từ đảo ngược. Mọi cặp được tìm từ góc nhìn của từ đứng TRƯỚC."""
    root = build_reversed_trie(words)
    result = []
    for i, w in enumerate(words):
        node, k = root, len(w)
        for j, ch in enumerate(w):
            # Tình huống 1: x = reverse(w[:j]) ngắn hơn w, cần w[j:] palindrome
            if node.word_id != -1 and node.word_id != i and is_pal_range(w, j, k - 1):
                result.append([i, node.word_id])
            node = node.children.get(ch)
            if node is None:
                break
        else:
            # Tình huống 2: đi hết w; mọi x trong pal_below đều ghép được
            for e in node.pal_below:
                if e != i:
                    result.append([i, e])
    return result


for words, expected in [(ex1, [[0, 1], [1, 0], [3, 2], [2, 4]]),
                        (ex2, [[0, 1], [1, 0]]),
                        (ex3, [[0, 1], [1, 0]])]:
    got = palindrome_pairs_trie(words)
    assert same_pairs(got, expected) and not has_duplicates(got)
    print(f"{words} -> {got}  PASS")

['abcd', 'dcba', 'lls', 's', 'sssll'] -> [[0, 1], [1, 0], [2, 4], [3, 2]]  PASS
['bat', 'tab', 'cat'] -> [[0, 1], [1, 0]]  PASS
['a', ''] -> [[0, 1], [1, 0]]  PASS


### Trace Cách 3 trên Ví dụ 1

`trace_trie` in từng bước đi xuống Trie của mỗi từ: độ sâu `j`, nút hiện tại (đường đi từ gốc), có `word_id` không, kiểm tra `w[j:]`, rồi ký tự đi tiếp. Khi đi hết từ thì in `pal_below` của nút cuối.

In [18]:
def trace_trie(words):
    root = build_reversed_trie(words)
    result = []
    for i, w in enumerate(words):
        node, k, path = root, len(w), ""
        print(f"--- w[{i}] = {w!r}")
        for j, ch in enumerate(w):
            note = ""
            if node.word_id != -1 and node.word_id != i:
                ok = is_pal_range(w, j, k - 1)
                note = f"word_id={node.word_id}, w[j:]={w[j:]!r} pal? {ok}"
                if ok:
                    result.append([i, node.word_id])
                    note += f" => THÊM {[i, node.word_id]}"
            nxt = node.children.get(ch)
            print(f"  j={j} nút={path or '(gốc)':<6} {note:<38} đi '{ch}': {'ok' if nxt else 'KHÔNG CÓ -> dừng'}")
            if nxt is None:
                break
            node, path = nxt, path + ch
        else:
            added = [[i, e] for e in node.pal_below if e != i]
            result.extend(added)
            print(f"  đi hết w ở nút {path!r}: pal_below={node.pal_below} => THÊM {added}")
    print("Kết quả:", result)
    return result


trace_trie(["abcd", "dcba", "lls", "s", "sssll"]);

--- w[0] = 'abcd'
  j=0 nút=(gốc)                                         đi 'a': ok
  j=1 nút=a                                             đi 'b': ok
  j=2 nút=ab                                            đi 'c': ok
  j=3 nút=abc                                           đi 'd': ok
  đi hết w ở nút 'abcd': pal_below=[1] => THÊM [[0, 1]]
--- w[1] = 'dcba'
  j=0 nút=(gốc)                                         đi 'd': ok
  j=1 nút=d                                             đi 'c': ok
  j=2 nút=dc                                            đi 'b': ok
  j=3 nút=dcb                                           đi 'a': ok
  đi hết w ở nút 'dcba': pal_below=[0] => THÊM [[1, 0]]
--- w[2] = 'lls'
  j=0 nút=(gốc)                                         đi 'l': ok
  j=1 nút=l                                             đi 'l': ok
  j=2 nút=ll                                            đi 's': ok
  đi hết w ở nút 'lls': pal_below=[4] => THÊM [[2, 4]]
--- w[3] = 's'
  j=0 nút=(gốc)             

### Đọc kết quả trace (Cách 3)

| Từ đứng trước `w` | Diễn biến | Cặp thêm |
|---|---|---|
| `w[0] = "abcd"` | đi hết `a→b→c→d` (chính là `reverse("dcba")`), nút cuối có `pal_below = [1]` | `[0, 1]` |
| `w[1] = "dcba"` | đi hết `d→c→b→a`, nút cuối có `pal_below = [0]` | `[1, 0]` |
| `w[2] = "lls"` | đi hết `l→l→s` — đó là 3 ký tự đầu của `reverse("sssll") = "llsss"`; phần còn lại `"ss"` là palindrome nên nút `lls` có `pal_below = [4]` | `[2, 4]` |
| `w[3] = "s"` | đi hết, nút `s` có `pal_below = [2, 3]`; bỏ 3 (chính nó) | `[3, 2]` |
| `w[4] = "sssll"` | ở nút `s` (`j = 1`) gặp `word_id = 3` (từ `"s"`) nhưng `w[1:] = "ssll"` không palindrome; nút `s` không có con `'s'` → **dừng sớm** | — |

Hai điều rút ra:

- Cách 3 tìm **mọi cặp từ góc nhìn của từ đứng trước** (`[i, ...]`). Cặp `[3, 2]` (`"s" + "lls"`) — mà Cách 2 bắt bằng Luật A khi đứng ở `"lls"` — ở đây được bắt khi đứng ở `"s"`, nhờ danh sách `pal_below` chuẩn bị sẵn.
- `w[4]` **dừng ngay** khi không còn nhánh: Trie tự động bỏ qua mọi lát cắt không có hy vọng, không cần tạo chuỗi `reverse(...)` để tra như Cách 2.

Ví dụ 1 chưa có ca **Tình huống 1** thành công. Cell dưới trace `["aba", "", "ab"]`: chuỗi rỗng `""` có đảo ngược kết thúc ngay ở **gốc** (`word_id` của gốc = 1), nên khi đứng ở `"aba"`, `j = 0`, ta gặp ngay `word_id = 1` và `w[0:] = "aba"` palindrome → `[0, 1]`.

In [19]:
got = trace_trie(["aba", "", "ab"])
assert same_pairs(got, palindrome_pairs_brute(["aba", "", "ab"]))

--- w[0] = 'aba'
  j=0 nút=(gốc)  word_id=1, w[j:]='aba' pal? True => THÊM [0, 1] đi 'a': ok
  j=1 nút=a                                             đi 'b': ok
  j=2 nút=ab                                            đi 'a': ok
  đi hết w ở nút 'aba': pal_below=[0] => THÊM []
--- w[1] = ''
  đi hết w ở nút '': pal_below=[0, 1] => THÊM [[1, 0]]
--- w[2] = 'ab'
  j=0 nút=(gốc)  word_id=1, w[j:]='ab' pal? False       đi 'a': ok
  j=1 nút=a                                             đi 'b': ok
  đi hết w ở nút 'ab': pal_below=[0] => THÊM [[2, 0]]
Kết quả: [[0, 1], [1, 0], [2, 0]]


### Độ phức tạp của Cách 3 — và một bài học về Python

Vẫn dùng `n`, `k`, `S` như mục 13.

- **Dựng Trie**: mỗi từ đi xuống `len(x)` bước, mỗi bước kiểm tra một palindrome `O(k)` → `O(n·k²)`. Số nút tối đa là `S + 1`.
- **Tìm cặp**: mỗi từ đi xuống tối đa `k` bước, mỗi bước có thể kiểm tra palindrome `O(k)` → `O(n·k²)`; cộng thêm số cặp in ra.
- **Bộ nhớ**: `O(S)` nút — với `S = 1.5 * 10^6` là tới **1.5 triệu object Python**, mỗi object kèm một `dict` riêng → có thể tốn hàng trăm MB.

Vậy về Big-O, Cách 3 **không tốt hơn** Cách 2: thừa số `k²` vẫn còn, do phép kiểm tra palindrome. Trie chỉ thay việc "băm cả chuỗi để tra" bằng "đi từng bước". Trong Python, mỗi bước đi bằng vòng lặp Python lại **chậm hơn nhiều** so với slicing/hash chạy bằng C ở Cách 2 — nên Cách 3 thường **chậm hơn** Cách 2 dù cùng Big-O. Mục 18 sẽ đo thử cả ba cách nhanh trên cùng dữ liệu (con số cụ thể tùy máy, nhưng xu hướng thường như vậy).

> Bài học: Big-O cho biết **tốc độ tăng** khi input lớn dần, không cho biết **hằng số**. Khi hai cách cùng Big-O, cách tận dụng thao tác "dựng sẵn" của Python thường thắng.

## 16. Kiến thức nền 6 — "Biên" (border) và mảng `lps` của KMP

Muốn bỏ thừa số `k` cuối cùng, ta cần trả lời câu hỏi "`w[:L]` có phải palindrome không?" cho **mọi** `L` của một từ chỉ trong **`O(k)` tổng cộng** (thay vì `O(k)` cho mỗi `L`). Công cụ: **mảng `lps`** trong thuật toán KMP.

**Biên** (border) của chuỗi `t` là một chuỗi vừa là **tiền tố** vừa là **hậu tố** của `t`, nhưng **ngắn hơn** `t`. Ví dụ `t = "aabaa"`: tiền tố `"a"`, `"aa"` cũng là hậu tố → các biên là `"aa"`, `"a"` (và chuỗi rỗng). `"aab"` không phải biên vì `t` không kết thúc bằng `"aab"`.

**`lps[i]`** (*longest proper prefix which is also suffix*) = độ dài biên **dài nhất** của `t[:i+1]`. Với `t = "aabaa"`:

| `i` | `t[:i+1]` | biên dài nhất | `lps[i]` |
|:-:|---|---|:-:|
| 0 | `a` | `""` | 0 |
| 1 | `aa` | `a` | 1 |
| 2 | `aab` | `""` | 0 |
| 3 | `aaba` | `a` | 1 |
| 4 | `aabaa` | `aa` | 2 |

Tính `lps` trong `O(m)` (`m = len(t)`) nhờ hai ý tưởng, với `j` = độ dài biên hiện tại (biên của `t[:i]`):

1. **Khớp thì kéo dài**: nếu `t[i] == t[j]` thì biên dài thêm 1: `j += 1`.
2. **Lệch thì lùi về "biên của biên"**: nếu `t[i] != t[j]`, thử biên dài nhất tiếp theo `j = lps[j - 1]`, lặp (`while`) đến khi khớp hoặc `j = 0`.

Vì sao chỉ `O(m)`? Mỗi vòng `for` tăng `j` **tối đa 1**, còn mỗi lần lùi trong `while` **giảm** `j` ít nhất 1. Tổng số lần giảm không vượt tổng số lần tăng, tức `< m`.

In [20]:
def compute_lps(t: str) -> list:
    """lps[i] = độ dài biên dài nhất của t[:i+1]. Chạy O(len(t))."""
    lps = [0] * len(t)
    j = 0                                   # độ dài biên hiện tại
    for i in range(1, len(t)):
        while j > 0 and t[i] != t[j]:
            j = lps[j - 1]                  # lệch: lùi về biên của biên
        if t[i] == t[j]:
            j += 1                          # khớp: kéo dài biên
        lps[i] = j
    return lps


def lps_by_definition(t: str) -> list:
    """Bản chậm theo đúng định nghĩa, dùng để đối chiếu."""
    out = []
    for i in range(len(t)):
        p = t[:i + 1]
        out.append(max(L for L in range(len(p)) if p[:L] == p[len(p) - L:]))
    return out


for t in ["aabaa", "abab", "aaaa", "abcab"]:
    print(f"{t!r:<8} lps = {compute_lps(t)}")
    assert compute_lps(t) == lps_by_definition(t)
print("compute_lps khớp định nghĩa.")

'aabaa'  lps = [0, 1, 0, 1, 2]
'abab'   lps = [0, 0, 1, 2]
'aaaa'   lps = [0, 1, 2, 3]
'abcab'  lps = [0, 0, 0, 1, 2]
compute_lps khớp định nghĩa.


## 17. Mẹo `w + "#" + reverse(w)`: mọi tiền tố palindrome trong `O(k)`

Đặt `t = w + "#" + reverse(w)` (dấu `"#"` không xuất hiện trong `w` vì `w` chỉ có chữ thường). Xét một biên dài `L` của `t`:

- `L` không thể vượt `k`: nếu dài hơn, tiền tố chứa `"#"` ở vị trí `k`, hậu tố lại chứa `"#"` ở vị trí khác → không thể bằng nhau.
- Tiền tố dài `L` của `t` là `w[:L]`. Hậu tố dài `L` của `t` là `L` ký tự cuối của `reverse(w)` — chính là `reverse(w[:L])`.
- Biên ⟺ `w[:L] == reverse(w[:L])` ⟺ **`w[:L]` là palindrome**.

Vậy: **các biên của `t` ↔ các tiền tố palindrome của `w`**. Còn liệt kê **mọi** biên thì dùng "chuỗi lùi": biên dài nhất là `L1 = lps[-1]`, biên dài nhất tiếp theo là `L2 = lps[L1 - 1]`, rồi `L3 = lps[L2 - 1]`... đến 0. (Lý do: một biên ngắn hơn của `t` thì cũng là biên của biên dài nhất — nó vừa là tiền tố, vừa là hậu tố của chuỗi đó.)

Ví dụ `w = "aabaa"`: tiền tố palindrome là `"a"`, `"aa"`, `"aabaa"` → độ dài `{1, 2, 5}` (và 0).

Hàm `pal_prefix_flags(w)` trả về list `ok` dài `k + 1`, với `ok[L] = True` ⟺ `w[:L]` là palindrome. Muốn biết **hậu tố** palindrome, chỉ cần gọi trên `reverse(w)`: `L` ký tự cuối của `w` là palindrome ⟺ `L` ký tự đầu của `reverse(w)` là palindrome.

In [21]:
def pal_prefix_flags(w: str) -> list:
    """ok[L] = True  <=>  w[:L] là palindrome, với L = 0..len(w). Chạy O(len(w))."""
    k = len(w)
    ok = [False] * (k + 1)
    ok[0] = True                              # tiền tố rỗng luôn là palindrome
    if k == 0:
        return ok
    lps = compute_lps(w + "#" + w[::-1])
    L = lps[-1]
    while L > 0:                              # đi theo chuỗi biên: L1, L2, ...
        ok[L] = True
        L = lps[L - 1]
    return ok


w = "aabaa"
print("t =", w + "#" + w[::-1], "| lps =", compute_lps(w + "#" + w[::-1]))
print("tiền tố palindrome của", repr(w), ":", [w[:L] for L, f in enumerate(pal_prefix_flags(w)) if f])
print("hậu tố  palindrome của 'lls'  :", [L for L, f in enumerate(pal_prefix_flags("lls"[::-1])) if f], "(độ dài)")

for w in ["", "a", "lls", "sssll", "abacaba", "abcd", "aabaa"]:
    assert pal_prefix_flags(w) == [is_palindrome(w[:L]) for L in range(len(w) + 1)]
print("pal_prefix_flags khớp cách kiểm tra trực tiếp.")

t = aabaa#aabaa | lps = [0, 1, 0, 1, 2, 0, 1, 2, 3, 4, 5]
tiền tố palindrome của 'aabaa' : ['', 'a', 'aa', 'aabaa']
hậu tố  palindrome của 'lls'  : [0, 1] (độ dài)
pal_prefix_flags khớp cách kiểm tra trực tiếp.


## 18. Cách 4 (nâng cao) — Trie + cờ palindrome tính sẵn: `O(tổng độ dài)`

Giữ nguyên Cách 3, chỉ thay **hai chỗ** gọi `is_pal_range` (mỗi chỗ tốn `O(k)`) bằng **tra cờ `O(1)`** đã tính sẵn cho từng từ:

| Chỗ cần hỏi | Ý nghĩa | Thay bằng |
|---|---|---|
| Lúc thêm `x` (độ dài `kx`), ở độ sâu `d`: `r[d:]` palindrome? (`r = reverse(x)`) | `r[d:] = reverse(x[:kx - d])` → palindrome ⟺ `x[:kx - d]` palindrome | `pre[e][kx - d]` |
| Lúc tìm cho `w` (độ dài `k`), ở độ sâu `j`: `w[j:]` palindrome? | đó là `k - j` ký tự cuối của `w` | `suf[i][k - j]` |

với `pre[e] = pal_prefix_flags(words[e])` và `suf[i] = pal_prefix_flags(reverse(words[i]))`.

**Độ phức tạp:** tính cờ `O(S)`; dựng Trie `O(S)`; tìm cặp `O(S)` cộng số cặp in ra (mỗi phần tử của `pal_below` được duyệt đều sinh ra một cặp, trừ tối đa một lần gặp chính nó). Tổng **`O(S + P)`** với `P` = số cặp kết quả — đúng tinh thần yêu cầu `O(sum of words[i].length)` của đề (phần `P` là bắt buộc với *bất kỳ* thuật toán nào, vì phải in ra từng cặp). Bộ nhớ `O(S)`.

In [22]:
def palindrome_pairs_trie_linear(words):
    """Cách 4 — Trie từ đảo ngược + cờ tiền tố/hậu tố palindrome tính bằng lps. O(S + số cặp)."""
    pre = [pal_prefix_flags(w) for w in words]          # pre[e][L]: words[e][:L] palindrome?
    suf = [pal_prefix_flags(w[::-1]) for w in words]    # suf[i][L]: L ký tự cuối của words[i] palindrome?

    root = TrieNode()
    for e, x in enumerate(words):
        kx, node = len(x), root
        for d, ch in enumerate(reversed(x)):            # duyệt reverse(x) mà không cần tạo chuỗi mới
            if pre[e][kx - d]:                          # phần còn lại reverse(x)[d:] là palindrome
                node.pal_below.append(e)
            node = node.children.setdefault(ch, TrieNode())
        node.word_id = e
        node.pal_below.append(e)

    result = []
    for i, w in enumerate(words):
        node, k = root, len(w)
        for j, ch in enumerate(w):
            if node.word_id != -1 and node.word_id != i and suf[i][k - j]:   # w[j:] palindrome
                result.append([i, node.word_id])
            node = node.children.get(ch)
            if node is None:
                break
        else:
            for e in node.pal_below:
                if e != i:
                    result.append([i, e])
    return result


for words, expected in [(ex1, [[0, 1], [1, 0], [3, 2], [2, 4]]),
                        (ex2, [[0, 1], [1, 0]]),
                        (ex3, [[0, 1], [1, 0]])]:
    got = palindrome_pairs_trie_linear(words)
    assert same_pairs(got, expected) and not has_duplicates(got)
    print(f"{words} -> {got}  PASS")

['abcd', 'dcba', 'lls', 's', 'sssll'] -> [[0, 1], [1, 0], [2, 4], [3, 2]]  PASS
['bat', 'tab', 'cat'] -> [[0, 1], [1, 0]]  PASS
['a', ''] -> [[0, 1], [1, 0]]  PASS


### So tốc độ ba cách nhanh trên hai kiểu dữ liệu

- **Ngẫu nhiên** (2000 từ dài 300): hầu như không có tiền tố/hậu tố palindrome dài, phép kiểm tra palindrome ở Cách 2 thường **dừng rất sớm** → cái giá `k²` gần như không xuất hiện.
- **"Nhiều palindrome"**: các từ dạng `"a"*p + "b" + "a"*q` (dài tới 300) — rất nhiều tiền tố/hậu tố là palindrome dài, mỗi lần kiểm tra phải so gần hết chuỗi → đây là kiểu dữ liệu làm lộ thừa số `k²`.

Hãy đọc kết quả với tinh thần của mục 15: Cách 4 có Big-O tốt nhất, nhưng trong Python, Cách 2 nhờ thao tác chuỗi bằng C vẫn thường rất cạnh tranh.

In [23]:
rng = random.Random(42)
ws_pal = sorted({"a" * p + "b" + "a" * q for p, q in ((rng.randint(0, 150), rng.randint(0, 149)) for _ in range(4000))})
ws_pal = ws_pal[:2000] + ["a" * L for L in range(0, 300, 3)]
datasets = [("ngẫu nhiên", random_words(2000, 300, seed=7)), ("nhiều palindrome", ws_pal)]
for label, ws in datasets:
    answers = []
    for name, f in [("Cách 2", palindrome_pairs_hashmap), ("Cách 3", palindrome_pairs_trie),
                    ("Cách 4", palindrome_pairs_trie_linear)]:
        start = time.perf_counter()
        r = f(ws)
        answers.append(sorted(r))
        print(f"{label:<16} {name}: {len(ws)} từ, {len(r):>6} cặp, {time.perf_counter() - start:.2f} giây")
    assert answers[0] == answers[1] == answers[2]

ngẫu nhiên       Cách 2: 2000 từ,      0 cặp, 0.31 giây


ngẫu nhiên       Cách 3: 2000 từ,      0 cặp, 0.99 giây


ngẫu nhiên       Cách 4: 2000 từ,      0 cặp, 1.20 giây


nhiều palindrome Cách 2: 2100 từ,  37626 cặp, 0.30 giây


nhiều palindrome Cách 3: 2100 từ,  37626 cặp, 0.77 giây


nhiều palindrome Cách 4: 2100 từ,  37626 cặp, 0.28 giây


### Đọc kết quả đo

- Trên dữ liệu **ngẫu nhiên**, Cách 3 và Cách 4 gần như ngang nhau (không có gì nhiều để tiết kiệm — phép kiểm tra palindrome vốn đã dừng sớm), và cả hai đều chậm hơn Cách 2 vì phải dựng hàng trăm nghìn nút Trie bằng vòng lặp Python.
- Trên dữ liệu **nhiều palindrome**, Cách 4 nhanh hơn Cách 3 rõ rệt — đó chính là thừa số `k` bị loại bỏ khi thay `is_pal_range` bằng tra cờ.
- Cách 2 vẫn rất nhanh trong cả hai trường hợp nhờ slicing/so sánh chuỗi chạy bằng C.

Kết luận thực dụng: **nộp LeetCode bằng Cách 2** (ngắn, dễ viết đúng, nhanh trong Python); hiểu Cách 3/4 để trả lời được câu hỏi "làm sao đạt `O(tổng độ dài)`?" khi phỏng vấn, và để dùng được Trie cho những bài mà hash map không làm được.

## 19. Code nộp LeetCode (Cách 2 — hash map)

Chuyển `palindrome_pairs_hashmap` sang đúng format LeetCode:

- `from typing import List` để dùng được chú thích kiểu `List[str]`, `List[List[int]]` như skeleton (trên LeetCode dòng này đã có sẵn; trong notebook ta phải tự import).
- Đổi tên biến cho gọn, bỏ mọi `print`.
- Một tối ưu nhỏ an toàn: chỉ tạo `reverse(...)` **khi** khúc tương ứng là palindrome (đã làm như vậy).

Chú ý các ca biên đã được lo sẵn bởi logic chung:

- `words = [""]` hoặc `words = ["a"]` (chỉ 1 từ) → không có cặp nào vì mọi lần tra đều gặp "chính nó" → `[]`.
- Có `""` trong danh sách → mọi từ palindrome khác ghép được 2 chiều với nó.

In [24]:
from typing import List


class Solution:
    def palindromePairs(self, words: List[str]) -> List[List[int]]:
        index = {w: i for i, w in enumerate(words)}
        res = []
        for i, w in enumerate(words):
            k = len(w)
            for j in range(k + 1):
                prefix, suffix = w[:j], w[j:]
                # Luật A: prefix palindrome  ->  reverse(suffix) + w
                if prefix == prefix[::-1]:
                    x = suffix[::-1]
                    if x in index and index[x] != i:
                        res.append([index[x], i])
                # Luật B: suffix palindrome  ->  w + reverse(prefix)   (j < k: tránh trùng)
                if j < k and suffix == suffix[::-1]:
                    x = prefix[::-1]
                    if x in index and index[x] != i:
                        res.append([i, index[x]])
        return res

In [25]:
sol = Solution()
tests = [
    (["abcd", "dcba", "lls", "s", "sssll"], [[0, 1], [1, 0], [3, 2], [2, 4]]),   # ví dụ 1
    (["bat", "tab", "cat"], [[0, 1], [1, 0]]),                                    # ví dụ 2
    (["a", ""], [[0, 1], [1, 0]]),                                                # ví dụ 3
    ([""], []),                                                                   # 1 từ rỗng
    (["abc"], []),                                                                # 1 từ
    (["aba"], []),                                                                # palindrome nhưng không được ghép với chính nó
    (["", "aba", "xy"], [[0, 1], [1, 0]]),                                        # chuỗi rỗng + palindrome
    (["a", "aa", "aaa"], [[0, 1], [1, 0], [0, 2], [2, 0], [1, 2], [2, 1]]),       # mọi cặp
    (["a", "b", "c", "ab", "ac", "aa"], [[3, 0], [1, 3], [4, 0], [2, 4], [5, 0], [0, 5]]),
    (["abc", "cba", "ba", "cb"], [[0, 1], [1, 0], [0, 2], [3, 0]]),                 # dài khác nhau
]
for words, expected in tests:
    got = sol.palindromePairs(words)
    assert same_pairs(got, expected), (words, got)
    assert same_pairs(got, palindrome_pairs_brute(words)) and not has_duplicates(got)
    print(f"{str(words):<38} -> {len(got)} cặp  PASS")
print()

# Input lớn nhất theo constraints: 5000 từ, mỗi từ dài 300
big_cases = [
    ("ngẫu nhiên 'ab'", random_words(5000, 300, seed=1)),
    ("ngẫu nhiên 26 chữ", random_words(5000, 300, alphabet="abcdefghijklmnopqrstuvwxyz", seed=2)),
    ("a...b...a dài 300 + toàn 'a'", sorted({"a" * p + "b" + "a" * (299 - p) for p in range(300)}
                                          | {"a" * p + "bb" + "a" * (298 - p) for p in range(299)})
                                   + ["a" * L for L in range(301)]),
]
for label, ws in big_cases:
    start = time.perf_counter()
    got = sol.palindromePairs(ws)
    elapsed = time.perf_counter() - start
    assert not has_duplicates(got)
    print(f"{label:<28}: n={len(ws)}, {len(got)} cặp, {elapsed:.2f} giây")

['abcd', 'dcba', 'lls', 's', 'sssll']  -> 4 cặp  PASS
['bat', 'tab', 'cat']                  -> 2 cặp  PASS
['a', '']                              -> 2 cặp  PASS
['']                                   -> 0 cặp  PASS
['abc']                                -> 0 cặp  PASS
['aba']                                -> 0 cặp  PASS
['', 'aba', 'xy']                      -> 2 cặp  PASS
['a', 'aa', 'aaa']                     -> 6 cặp  PASS
['a', 'b', 'c', 'ab', 'ac', 'aa']      -> 6 cặp  PASS
['abc', 'cba', 'ba', 'cb']             -> 4 cặp  PASS



ngẫu nhiên 'ab'             : n=5000, 0 cặp, 0.45 giây


ngẫu nhiên 26 chữ           : n=5000, 0 cặp, 0.45 giây
a...b...a dài 300 + toàn 'a': n=900, 91498 cặp, 0.15 giây


### (Tùy chọn) Bản LeetCode của Cách 3 — Trie

Nếu muốn nộp bằng Trie, đổi tên `SolutionTrie` thành `Solution`. Code tự chứa (không gọi hàm ngoài class), dùng hàm lồng `is_pal(s, lo, hi)`.

In [26]:
class SolutionTrie:
    def palindromePairs(self, words: List[str]) -> List[List[int]]:
        def is_pal(s, lo, hi):
            while lo < hi:
                if s[lo] != s[hi]:
                    return False
                lo += 1
                hi -= 1
            return True

        # Mỗi nút: [children(dict), word_id, pal_below(list)] — dùng list cho nhẹ
        root = [{}, -1, []]
        for e, x in enumerate(words):
            r, node = x[::-1], root
            for d, ch in enumerate(r):
                if is_pal(r, d, len(r) - 1):
                    node[2].append(e)
                node = node[0].setdefault(ch, [{}, -1, []])
            node[1] = e
            node[2].append(e)

        res = []
        for i, w in enumerate(words):
            node, k = root, len(w)
            for j, ch in enumerate(w):
                if node[1] != -1 and node[1] != i and is_pal(w, j, k - 1):
                    res.append([i, node[1]])
                node = node[0].get(ch)
                if node is None:
                    break
            else:
                res.extend([i, e] for e in node[2] if e != i)
        return res


for words, expected in tests:
    assert same_pairs(SolutionTrie().palindromePairs(words), expected), words
print("SolutionTrie PASS", len(tests), "test")

SolutionTrie PASS 10 test


## 20. Đối chiếu chéo tất cả các cách trên input ngẫu nhiên

Test tay chỉ bao được vài trường hợp. Ta sinh **3000 bộ `words` ngẫu nhiên** và yêu cầu mọi cách cho **cùng tập cặp** với brute force, đồng thời **không có cặp trùng**:

- `random.seed(336)` cố định dữ liệu → chạy lại vẫn ra đúng như cũ, dễ tái hiện lỗi.
- Bảng chữ cái **nhỏ** (`"ab"` hoặc `"abc"`) và từ **ngắn** (0–6 ký tự) để có thật nhiều palindrome, nhiều chuỗi rỗng, nhiều cặp "đảo ngược của nhau" — các ca dễ sai nhất.
- `words` được lấy qua `set` để bảo đảm các từ **khác nhau** đúng như đề, rồi `sorted` để thứ tự cố định.

In [27]:
random.seed(336)
methods = {
    "hashmap": palindrome_pairs_hashmap,
    "trie": palindrome_pairs_trie,
    "trie_linear": palindrome_pairs_trie_linear,
    "Solution": Solution().palindromePairs,
    "SolutionTrie": SolutionTrie().palindromePairs,
}
total_pairs = 0
for _ in range(3000):
    alphabet = random.choice(["ab", "abc"])
    n = random.randint(1, 9)
    words = sorted({"".join(random.choice(alphabet) for _ in range(random.randint(0, 6))) for _ in range(n)})
    expected = sorted(palindrome_pairs_brute(words))
    total_pairs += len(expected)
    for name, f in methods.items():
        got = f(words)
        assert sorted(got) == expected, (name, words, got)
        assert not has_duplicates(got), (name, words)
print(f"3000 bộ words ngẫu nhiên: {len(methods)} cách khớp brute force, tổng cộng {total_pairs} cặp, không trùng.")

3000 bộ words ngẫu nhiên: 5 cách khớp brute force, tổng cộng 9186 cặp, không trùng.


## 21. Vì sao đúng? Chứng minh "không thừa, không sót, không trùng" cho Cách 2

**Không thừa (mọi cặp thêm vào đều đúng).** Luật A thêm `[x, w]` khi `w = prefix + suffix`, `prefix` palindrome, `x = reverse(suffix)`: khi đó `x + w = reverse(suffix) + prefix + suffix` — đọc ngược lại vẫn là chính nó. Luật B tương tự với `w + x = prefix + suffix + reverse(prefix)`. Điều kiện `index[x] != i` bảo đảm hai chỉ số khác nhau.

**Không sót.** Lấy một cặp đúng bất kỳ `[a, b]`, `A = words[a]` đứng trước, `B = words[b]` đứng sau. Theo mục 7:

- Nếu `len(A) > len(B)`: `A = reverse(B) + P` với `P` palindrome. Khi đứng ở `A`, lát cắt `j = len(B)` (thỏa `j < len(A)`) có `prefix = reverse(B)`, `suffix = P` palindrome → Luật B tìm `reverse(prefix) = B` → thêm `[a, b]`.
- Nếu `len(A) <= len(B)`: `B = P + reverse(A)` với `P` palindrome (khi bằng nhau thì `P = ""`). Khi đứng ở `B`, lát cắt `j = len(B) - len(A)` có `prefix = P`, `suffix = reverse(A)` → Luật A tìm `reverse(suffix) = A` → thêm `[a, b]`.

**Không trùng.** Mỗi lần thêm `[a, b]` thì xảy ra theo đúng một trong hai kiểu trên và lát cắt bị **ép** bởi độ dài:

- Luật B đứng ở `A`, cắt tại `j = len(B)`, và vì đòi `j < len(A)` nên chỉ xảy ra khi `len(A) > len(B)`.
- Luật A đứng ở `B`, cắt tại `j = len(B) - len(A) >= 0`, nên chỉ xảy ra khi `len(A) <= len(B)`.

Hai điều kiện độ dài loại trừ nhau, và trong mỗi kiểu chỉ có **một** lát cắt khả dĩ → mỗi cặp được thêm **đúng một lần**. Đây chính là lý do điều kiện `j < k` ở Luật B là đủ — và cần (thiếu nó, ca `len(A) == len(B)` bị đếm hai lần như mục 10).

**Invariant bằng lời:** gọi **"nơi sinh"** của một cặp đúng `[a, b]` là từ `A` nếu `len(A) > len(B)`, ngược lại là từ `B`. Khi đó: *sau khi Cách 2 xử lý xong các từ `0, 1, ..., i`, `result` chứa đúng những cặp đúng có nơi sinh nằm trong các từ đó — mỗi cặp đúng một lần.* Xử lý hết `n` từ là có đủ đáp án.

Cell dưới kiểm chứng bằng thực nghiệm: ghi lại "nơi sinh" của từng cặp (đứng ở từ nào, luật nào, `j` bao nhiêu) và so với dự đoán theo độ dài.

In [28]:
def pairs_with_origin(words):
    """Như Cách 2 nhưng trả về (cặp, từ đang đứng, luật, j)."""
    index = {w: i for i, w in enumerate(words)}
    out = []
    for i, w in enumerate(words):
        k = len(w)
        for j in range(k + 1):
            prefix, suffix = w[:j], w[j:]
            if is_palindrome(prefix) and index.get(suffix[::-1], i) != i:
                out.append(((index[suffix[::-1]], i), i, "A", j))
            if j < k and is_palindrome(suffix) and index.get(prefix[::-1], i) != i:
                out.append(((i, index[prefix[::-1]]), i, "B", j))
    return out


words = ["aba", "", "ab", "ba", "abaab"]
for (a, b), at, rule, j in pairs_with_origin(words):
    A, B = words[a], words[b]
    predicted = ("B", a, len(B)) if len(A) > len(B) else ("A", b, len(B) - len(A))
    assert (rule, at, j) == predicted
    print(f"[{a},{b}] {A!r:>7} + {B!r:<7} | len {len(A)} vs {len(B)} -> Luật {rule} đứng ở w[{at}], j={j}  (khớp dự đoán)")

[0,1]   'aba' + ''      | len 3 vs 0 -> Luật B đứng ở w[0], j=0  (khớp dự đoán)
[2,0]    'ab' + 'aba'   | len 2 vs 3 -> Luật A đứng ở w[0], j=1  (khớp dự đoán)
[0,3]   'aba' + 'ba'    | len 3 vs 2 -> Luật B đứng ở w[0], j=2  (khớp dự đoán)
[1,0]      '' + 'aba'   | len 0 vs 3 -> Luật A đứng ở w[0], j=3  (khớp dự đoán)
[3,2]    'ba' + 'ab'    | len 2 vs 2 -> Luật A đứng ở w[2], j=0  (khớp dự đoán)
[2,3]    'ab' + 'ba'    | len 2 vs 2 -> Luật A đứng ở w[3], j=0  (khớp dự đoán)
[3,4]    'ba' + 'abaab' | len 2 vs 5 -> Luật A đứng ở w[4], j=3  (khớp dự đoán)


### Bảng so sánh các cách

`n` = số từ, `k` = độ dài tối đa, `S` = tổng độ dài, `P` = số cặp kết quả.

| Cách | Ý tưởng | Thời gian | Bộ nhớ thêm | Ưu điểm | Nhược điểm |
|---|---|---|---|---|---|
| 1. Brute force | thử mọi cặp `(i, j)` | `O(n²·k)` | `O(k)` | dễ nhất, làm oracle | `n = 5000` quá chậm (cỡ chục giây trở lên) |
| 2. Hash map + cắt từ | 2 luật prefix/suffix, tra `dict` | `O(n·k²)` | `O(S)` | ngắn, nhanh nhất trong Python, **nên nộp** | phải nhớ `j < k`, `!= i` |
| 3. Trie từ đảo ngược | đi `w` xuống Trie, `pal_below` | `O(n·k²)` | `O(S)` nút | dừng sớm khi hết nhánh; nền cho Cách 4 | code dài, nhiều object, chậm hơn trong Python |
| 4. Trie + cờ `lps` | cờ tiền tố/hậu tố palindrome tính sẵn | `O(S + P)` | `O(S)` | đạt đúng yêu cầu `O(tổng độ dài)` | khó nhất, cần thêm KMP |

## 22. Lỗi thường gặp

### Lỗi 1: Quên `j < k` ở Luật B → cặp trùng
Đã thấy ở mục 10: hai từ dài bằng nhau và là đảo ngược của nhau (`"abcd"`, `"dcba"`) bị đếm hai lần. **Sửa:** Luật B chỉ chạy khi `j < k` (hoặc tương đương: Luật A chỉ chạy khi `j > 0` — chọn **một** trong hai, đừng chặn cả hai, sẽ **sót**).

### Lỗi 2: Quên `index[x] != i` → ghép từ với chính nó
Mọi từ palindrome `w` có `reverse(w) = w`, nên lát cắt `j = 0` sẽ "tìm thấy" chính nó → thêm `[i, i]`.

### Lỗi 3: Đặt sai thứ tự cặp
Luật A: từ tìm được đứng **trước** (`[index[x], i]`); Luật B: đứng **sau** (`[i, index[x]]`). Đảo lẫn → trả về những cặp mà phép nối **không** phải palindrome (ví dụ `[2, 3]` = `"lls" + "s"`).

### Lỗi 4: `range(k)` thay vì `range(k + 1)`
Mất lát cắt `j = k` (prefix là cả từ, suffix rỗng) → **sót** các cặp `["" , w]` với `w` palindrome. Ví dụ 3 chỉ còn trả về 1 cặp.

### Lỗi 5 (Trie): quên thêm `e` vào `pal_below` ở **nút cuối**
"Phần còn lại rỗng" cũng là palindrome. Quên dòng này thì mất mọi cặp hai từ **dài bằng nhau** (`"abcd" + "dcba"`) và cặp `"" + w`.

### Lỗi 6: So kết quả bằng `==` mà không sắp xếp
Đáp án đúng nhưng khác thứ tự sẽ bị báo sai. Luôn so `sorted(a) == sorted(b)`.

### Lỗi 7: Tra cứu bằng `words.index(x)` hoặc vòng `for` tìm trong list
Mỗi lần tra thành `O(n)` → tổng `O(n²·k)`, mất hết lợi ích. Phải dùng `dict`.

Cell dưới chạy các bản lỗi 2, 3, 4, 5 để bạn tận mắt thấy hậu quả.

In [29]:
def bug_self_pair(words):              # Lỗi 2: không chặn index[x] != i
    index = {w: i for i, w in enumerate(words)}
    res = []
    for i, w in enumerate(words):
        for j in range(len(w) + 1):
            p, s = w[:j], w[j:]
            if is_palindrome(p) and s[::-1] in index:
                res.append([index[s[::-1]], i])
            if j < len(w) and is_palindrome(s) and p[::-1] in index:
                res.append([i, index[p[::-1]]])
    return res


def bug_wrong_order(words):            # Lỗi 3: Luật A đặt w đứng trước
    index = {w: i for i, w in enumerate(words)}
    res = []
    for i, w in enumerate(words):
        for j in range(len(w) + 1):
            p, s = w[:j], w[j:]
            if is_palindrome(p) and index.get(s[::-1], i) != i:
                res.append([i, index[s[::-1]]])                 # SAI: phải là [index[x], i]
            if j < len(w) and is_palindrome(s) and index.get(p[::-1], i) != i:
                res.append([i, index[p[::-1]]])
    return res


def bug_range_k(words):                # Lỗi 4: range(k)
    index = {w: i for i, w in enumerate(words)}
    res = []
    for i, w in enumerate(words):
        for j in range(len(w)):                                 # SAI: thiếu j = k
            p, s = w[:j], w[j:]
            if is_palindrome(p) and index.get(s[::-1], i) != i:
                res.append([index[s[::-1]], i])
            if is_palindrome(s) and index.get(p[::-1], i) != i:
                res.append([i, index[p[::-1]]])
    return res


print("Lỗi 2, ['aba', 'x']        :", bug_self_pair(["aba", "x"]), " (đúng phải là [])")
wrong = bug_wrong_order(ex1)
print("Lỗi 3, Ví dụ 1             :", wrong, "-> có cặp sai:",
      [p for p in wrong if not is_palindrome(ex1[p[0]] + ex1[p[1]])])
print("Lỗi 4, Ví dụ 3 ['a', '']   :", bug_range_k(ex3), " (đúng phải có 2 cặp)")

# Lỗi 5: dựng Trie quên dòng node.pal_below.append(e) ở nút cuối
real_build = build_reversed_trie
def build_forgot_end(words):
    root = real_build(words)
    stack = [root]
    while stack:                                   # gỡ e khỏi pal_below ở nút cuối của chính nó
        node = stack.pop()
        if node.word_id != -1:
            node.pal_below.remove(node.word_id)
        stack.extend(node.children.values())
    return root
build_reversed_trie = build_forgot_end
print("Lỗi 5, Ví dụ 1 (Trie)      :", palindrome_pairs_trie(ex1), " (mất [0,1], [1,0])")
build_reversed_trie = real_build                   # trả lại bản đúng
assert same_pairs(palindrome_pairs_trie(ex1), [[0, 1], [1, 0], [3, 2], [2, 4]])

Lỗi 2, ['aba', 'x']        : [[0, 0], [1, 1]]  (đúng phải là [])
Lỗi 3, Ví dụ 1             : [[0, 1], [1, 0], [2, 3], [4, 2]] -> có cặp sai: [[2, 3], [4, 2]]
Lỗi 4, Ví dụ 3 ['a', '']   : [[0, 1]]  (đúng phải có 2 cặp)
Lỗi 5, Ví dụ 1 (Trie)      : [[2, 4], [3, 2]]  (mất [0,1], [1,0])


## 23. Mẹo nhận dạng pattern

- **"Ghép hai phần tử để được thứ có tính chất X"** + `n` lớn → đừng thử mọi cặp. Hỏi: *biết phần tử thứ nhất, phần tử thứ hai **phải là gì**?* Rồi tra trong **hash map**. Đây là "Two Sum cho chuỗi": `needed = target - x` ở Two Sum ↔ `x = reverse(suffix)` ở bài này.
- **Constraints lệch**: `n` lớn (5000) nhưng `k` nhỏ (300) → làm việc `O(k)` hay `O(k²)` **bên trong** mỗi phần tử, tránh `O(n²)` giữa các phần tử.
- **Palindrome của phép nối `A + B`** → nghĩ ngay tới "từ dài hơn = gương của từ ngắn hơn + khúc giữa palindrome" và **cắt từ thành prefix | suffix ở mọi vị trí**.
- **Nhiều truy vấn kiểu "có từ nào bắt đầu / kết thúc bằng …"** → Trie (cho "kết thúc bằng" thì Trie trên **chuỗi đảo ngược**).
- **Cần biết mọi tiền tố (hoặc hậu tố) palindrome của một chuỗi** → mẹo `s + "#" + reverse(s)` + mảng `lps` (cũng là chìa khóa của bài #214 Shortest Palindrome).
- **Đáp án "thứ tự bất kỳ"** → khi test, luôn sắp xếp trước khi so; kiểm tra thêm "không trùng".

---

## 24. Biến thể và bài liên quan

| Biến thể / bài | Thay đổi gì so với bài này |
|---|---|
| Chỉ **đếm** số cặp | thay `append` bằng `count += 1` |
| Cặp **không thứ tự** (`{i, j}`), chỉ đếm một lần | lưu `(min(i, j), max(i, j))` vào `set` — vì có thể cả `A + B` lẫn `B + A` đều palindrome |
| `words` **có thể trùng nhau** | `dict` từ → **list chỉ số** (Bài tập 5) |
| Trả về **chính các palindrome** tạo được | `words[i] + words[j]` cho từng cặp |
| LeetCode **#1** Two Sum | cùng tư duy "tính phần còn thiếu rồi tra hash map" |
| LeetCode **#214** Shortest Palindrome | tiền tố palindrome dài nhất, mẹo `s + "#" + reverse(s)` + `lps` |
| LeetCode **#5** Longest Palindromic Substring | kiểm tra/mở rộng palindrome quanh tâm |
| LeetCode **#131** Palindrome Partitioning | cắt chuỗi ở mọi vị trí + kiểm tra palindrome từng khúc |
| LeetCode **#208** Implement Trie | luyện riêng cấu trúc Trie |
| LeetCode **#2131** Longest Palindrome by Concatenating Two Letter Words | ghép các từ 2 chữ cái thành palindrome dài nhất (Bài tập 6) |

## 25. Bài tập tự luyện

Làm theo thứ tự — bài sau dùng lại kỹ năng của bài trước. Gợi ý theo 3 mức ở mục 26, đáp án ở mục 27 (cuối notebook). **Các cell TODO chỉ định nghĩa hàm, không tự chạy test** — làm xong hãy bỏ dấu `#` trước các dòng `assert`.

### Bài 1 — Đoán output (không chạy code)

Cho `words = ["ab", "ba", "a"]`.

- (a) Liệt kê mọi cặp palindrome (thử tay cả 6 cặp có thứ tự).
- (b) Khi Cách 2 đứng ở `w = "ab"` (chỉ số 0), những lát cắt `j` nào thêm cặp, theo luật nào, cặp gì?
- (c) `palindrome_pairs_hashmap_naive(words)` (bản **thiếu** `j < k` ở mục 10) trả về bao nhiêu phần tử? Cặp nào bị lặp?

### Bài 2 — Tính tay bảng lát cắt

Cho `words = ["aab", "baa", "b", ""]`. Lập bảng như mục 8 cho `w = "aab"` (chỉ số 0): với mỗi `j = 0..3`, ghi `prefix`, `suffix`, luật nào áp dụng, chuỗi cần tìm, có hay không, cặp thêm vào. Sau đó đoán **toàn bộ** đáp án của `words`.

---

### Bài 3 — Điền TODO: lời giải hash map

Hoàn thiện thân vòng lặp theo hai luật của mục 10–11. Đừng quên hai điều kiện chặn.

In [30]:
def exercise_palindrome_pairs(words):
    index = {w: i for i, w in enumerate(words)}
    result = []
    for i, w in enumerate(words):
        k = len(w)
        for j in range(k + 1):
            prefix, suffix = w[:j], w[j:]
            # TODO 1 (Luật A): nếu prefix là palindrome, tìm x = reverse(suffix);
            #         nếu x có trong index và index[x] != i -> thêm [index[x], i]
            # TODO 2 (Luật B): nếu j < k và suffix là palindrome, tìm x = reverse(prefix);
            #         nếu x có trong index và index[x] != i -> thêm [i, index[x]]
            pass
    return result


# Tự làm xong rồi mới mở test:
# assert same_pairs(exercise_palindrome_pairs(["abcd", "dcba", "lls", "s", "sssll"]), [[0, 1], [1, 0], [3, 2], [2, 4]])
# assert same_pairs(exercise_palindrome_pairs(["bat", "tab", "cat"]), [[0, 1], [1, 0]])
# assert same_pairs(exercise_palindrome_pairs(["a", ""]), [[0, 1], [1, 0]])
# assert not has_duplicates(exercise_palindrome_pairs(["abcd", "dcba", "lls", "s", "sssll"]))

### Bài 4 — Tự viết lại lời giải Trie

Viết lại Cách 3 trong **một hàm**, tự định nghĩa nút Trie (có thể dùng `dict`/`list` thay class). Cố gắng **không nhìn** mục 15. Hai việc chính: (1) dựng Trie trên các từ đảo ngược kèm `word_id`, `pal_below`; (2) đi từng từ xuống Trie với hai tình huống.

In [31]:
def exercise_palindrome_pairs_trie(words):
    # TODO 1: dựng Trie trên reverse(x) cho mọi x; ở độ sâu d, nếu phần còn lại là palindrome
    #         thì thêm chỉ số vào pal_below; nút cuối: gán word_id và thêm vào pal_below
    # TODO 2: với mỗi w (chỉ số i), đi xuống theo từng ký tự:
    #         - gặp word_id != -1, != i và w[j:] palindrome -> thêm [i, word_id]
    #         - hết nhánh thì dừng
    # TODO 3: nếu đi hết w: thêm [i, e] cho mọi e trong pal_below của nút cuối, e != i
    raise NotImplementedError


# assert same_pairs(exercise_palindrome_pairs_trie(["abcd", "dcba", "lls", "s", "sssll"]), [[0, 1], [1, 0], [3, 2], [2, 4]])
# assert same_pairs(exercise_palindrome_pairs_trie(["a", ""]), [[0, 1], [1, 0]])
# assert same_pairs(exercise_palindrome_pairs_trie(["aba", "", "ab"]), [[0, 1], [1, 0], [2, 0]])

### Bài 5 — Biến thể: `words` có thể có từ **trùng nhau**

Giả sử bỏ điều kiện "các từ đôi một khác nhau". Vẫn trả về mọi cặp chỉ số `(i, j)`, `i != j`, mà `words[i] + words[j]` là palindrome.

| `words` | Kết quả (thứ tự bất kỳ) |
|---|---|
| `["a", "a", ""]` | 6 cặp: mọi cặp `i != j` |
| `["ab", "ba", "ba"]` | `[[0,1],[1,0],[0,2],[2,0]]` |
| `["abc", "abc"]` | `[]` |

Câu hỏi phụ: vì sao bản `dict` "từ → một chỉ số" của Cách 2 không còn đúng? Hai từ trùng nhau `"a"`, `"a"` có ghép được với nhau không, và cặp đó được Luật nào bắt?

In [32]:
def exercise_palindrome_pairs_with_duplicates(words):
    """words có thể trùng nhau."""
    # TODO 1: index = dict: từ -> LIST các chỉ số (gợi ý: collections.defaultdict(list))
    # TODO 2: giữ nguyên 2 luật cắt; mỗi lần tìm thấy x, duyệt mọi chỉ số t trong index[x] với t != i
    raise NotImplementedError


# assert same_pairs(exercise_palindrome_pairs_with_duplicates(["a", "a", ""]), palindrome_pairs_brute(["a", "a", ""]))
# assert same_pairs(exercise_palindrome_pairs_with_duplicates(["ab", "ba", "ba"]), [[0, 1], [1, 0], [0, 2], [2, 0]])
# assert exercise_palindrome_pairs_with_duplicates(["abc", "abc"]) == []

### Bài 6 — Mini challenge: LeetCode 2131 *Longest Palindrome by Concatenating Two Letter Words*

Cho mảng `words`, mỗi từ gồm **đúng 2** chữ cái thường (từ **có thể trùng**). Chọn một số từ (mỗi phần tử dùng tối đa một lần) và **sắp xếp theo thứ tự tùy ý** rồi nối lại để được một palindrome. Trả về **độ dài lớn nhất** có thể (không chọn được thì trả về 0). Constraints: `1 <= words.length <= 10^5`, `words[i].length == 2`.

| `words` | Kết quả | Giải thích |
|---|:-:|---|
| `["lc","cl","gg"]` | 6 | `"lc" + "gg" + "cl" = "lcggcl"` |
| `["ab","ty","yt","lc","cl","ab"]` | 8 | `"tylcclyt"` |
| `["cc","ll","xx"]` | 2 | chỉ một từ "đôi" đứng giữa, ví dụ `"cc"` |

Gợi ý công cụ: `collections.Counter(words)` đếm số lần xuất hiện của mỗi từ, ví dụ `Counter(["ab","ab","cl"]) = {"ab": 2, "cl": 1}`; truy cập khóa không có thì trả về 0 (không lỗi).

In [33]:
from collections import Counter


def exercise_longest_palindrome_two_letters(words):
    # TODO 1: cnt = Counter(words)
    # TODO 2: với từ w = "xy" (x != y): ghép được min(cnt[w], cnt["yx"]) cặp, mỗi cặp +4 (đừng đếm 2 lần!)
    # TODO 3: với từ "xx": ghép (cnt // 2) cặp, mỗi cặp +4; nếu còn lẻ thì có thể đặt MỘT từ "xx" ở giữa (+2)
    raise NotImplementedError


# assert exercise_longest_palindrome_two_letters(["lc", "cl", "gg"]) == 6
# assert exercise_longest_palindrome_two_letters(["ab", "ty", "yt", "lc", "cl", "ab"]) == 8
# assert exercise_longest_palindrome_two_letters(["cc", "ll", "xx"]) == 2

## 26. Gợi ý theo mức

**Bài 1**
- Mức 1: chỉ có 6 cặp có thứ tự, cứ nối thử từng cặp.
- Mức 2: với (b), `"ab"` có 3 lát cắt `j = 0, 1, 2`; kiểm tra prefix/suffix nào là palindrome.
- Mức 3: với (c), bản thiếu `j < k` đếm trùng đúng những cặp hai từ **dài bằng nhau** — ở đây là `"ab"`/`"ba"`.

**Bài 2**
- Mức 1: `prefix` palindrome ở `j = 0` (rỗng), `j = 1` (`"a"`), `j = 2` (`"aa"`).
- Mức 2: Luật A ở `j = 0` tìm `reverse("aab") = "baa"`; ở `j = 2` tìm `reverse("b") = "b"`.
- Mức 3: toàn bộ đáp án: nhớ chuỗi rỗng `""` ghép 2 chiều với mọi từ palindrome (ở đây chỉ `"b"`).

**Bài 3**
- Mức 1: Luật A → cặp `[x, w]`; Luật B → cặp `[w, x]`.
- Mức 2: hai điều kiện chặn: `index[x] != i` (không ghép với chính mình) và `j < k` ở Luật B (không trùng).
- Mức 3: `if prefix == prefix[::-1]: x = suffix[::-1]; if x in index and index[x] != i: result.append([index[x], i])` — Luật B đối xứng.

**Bài 4**
- Mức 1: nút Trie có thể là list `[children, word_id, pal_below]`.
- Mức 2: lúc dựng, kiểm tra `r[d:]` palindrome **trước** khi đi theo `r[d]`; nút cuối luôn thêm chỉ số vào `pal_below`.
- Mức 3: lúc tìm, dùng `for ... else`: khối `else` (đi hết `w`) mới duyệt `pal_below`.

**Bài 5**
- Mức 1: chứng minh "không sót, không trùng" ở mục 21 chỉ dựa vào **độ dài** và **lát cắt**, không dựa vào việc từ khác nhau.
- Mức 2: `index = defaultdict(list)`; `index[w].append(i)`. Tìm thấy `x` thì có thể có **nhiều** chỉ số.
- Mức 3: hai từ `"a"` trùng nhau: đứng ở từ thứ hai (`len` bằng nhau), Luật A ở `j = 0` tìm `"a"` → có chỉ số của từ thứ nhất (khác `i`) → thêm cặp.

**Bài 6**
- Mức 1: palindrome ghép từ các khối 2 chữ: khối ở nửa trái `"xy"` phải có khối "gương" `"yx"` ở nửa phải.
- Mức 2: với `x != y`, mỗi cặp `("xy", "yx")` đóng góp 4 ký tự; chỉ duyệt một chiều (ví dụ khi `w < w[::-1]`) để không đếm 2 lần.
- Mức 3: từ `"xx"` tự là gương của chính nó: `cnt // 2` cặp; nếu có **bất kỳ** từ `"xx"` nào số lượng lẻ thì +2 (chỉ một lần cho cả bài).

## 27. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1** — `words = ["ab", "ba", "a"]`
- (a) `"ab"+"ba" = "abba"` ✔ `[0,1]`; `"ba"+"ab" = "baab"` ✔ `[1,0]`; `"ab"+"a" = "aba"` ✔ `[0,2]`; `"a"+"ba" = "aba"` ✔ `[2,1]`; còn `"a"+"ab" = "aab"` ✘, `"ba"+"a" = "baa"` ✘. → **4 cặp**.
- (b) Đứng ở `"ab"`: `j = 0` (`prefix = ""`): Luật A tìm `"ba"` → thêm `[1, 0]`. `j = 1` (`prefix = "a"`, `suffix = "b"`): Luật A tìm `"b"` → không có; Luật B tìm `reverse("a") = "a"` → thêm `[0, 2]`. `j = 2`: `prefix = "ab"` không palindrome; Luật B bị chặn vì `j = k`.
- (c) **6 phần tử**: `[1,0]` và `[0,1]` mỗi cặp bị lặp 2 lần (hai từ dài bằng nhau `"ab"`, `"ba"`).

**Bài 2** — `w = "aab"` trong `["aab", "baa", "b", ""]`:

| `j` | `prefix` | `suffix` | Luật | Tìm | Có? | Cặp thêm |
|:-:|---|---|:-:|---|:-:|---|
| 0 | `""` | `"aab"` | A | `"baa"` | ✔ [1] | `[1, 0]` (`"baaaab"`) |
| 1 | `"a"` | `"ab"` | A | `"ba"` | ✘ | — |
| 2 | `"aa"` | `"b"` | A | `"b"` | ✔ [2] | `[2, 0]` (`"baab"`) |
| 2 | | | B | `"aa"` | ✘ | — |
| 3 | `"aab"` | `""` | B | (bỏ qua vì `j = k`) | | — |

Toàn bộ đáp án: `[[0,1],[1,0],[1,2],[2,0],[2,3],[3,2]]` — tức `"aab"+"baa"`, `"baa"+"aab"`, `"baa"+"b"`, `"b"+"aab"`, `"b"+""`, `""+"b"`.

**Bài 5 — câu hỏi phụ.** Với `dict` "từ → **một** chỉ số", hai từ giống nhau sẽ **ghi đè** lên nhau: `{w: i for ...}` trên `["a", "a", ""]` chỉ còn `{"a": 1, "": 2}` — chỉ số 0 biến mất, các cặp chứa nó bị sót. Hai từ `"a"`, `"a"` **ghép được** với nhau (`"aa"` là palindrome và hai chỉ số khác nhau): đứng ở từ 0, Luật A với `j = 0` tìm `reverse("a") = "a"` → gặp chỉ số 1 → thêm `[1, 0]`; đứng ở từ 1 thì thêm `[0, 1]`. Chứng minh ở mục 21 vẫn đúng nguyên vẹn vì nó chỉ dựa vào độ dài và lát cắt.

**Bài 6 — ý chính.** Trong palindrome ghép từ các khối 2 chữ, khối thứ `t` từ trái phải là gương của khối thứ `t` từ phải. Khối `"xy"` (`x != y`) chỉ đi cặp được với `"yx"`; khối `"xx"` đi cặp với một `"xx"` khác, hoặc đứng **một mình ở chính giữa** (tối đa một khối như vậy).

Cell dưới kiểm chứng Bài 1–2 bằng code; các cell sau là đáp án Bài 3–4, Bài 5, Bài 6.

In [34]:
# ---------- Kiểm chứng Bài 1 ----------
w1 = ["ab", "ba", "a"]
assert same_pairs(palindrome_pairs_brute(w1), [[0, 1], [1, 0], [0, 2], [2, 1]])
naive = palindrome_pairs_hashmap_naive(w1)
print("Bài 1(c): naive trả về", naive, "->", len(naive), "phần tử")
assert len(naive) == 6

# ---------- Kiểm chứng Bài 2 ----------
w2 = ["aab", "baa", "b", ""]
assert same_pairs(palindrome_pairs_hashmap(w2), [[0, 1], [1, 0], [1, 2], [2, 0], [2, 3], [3, 2]])
print("Bài 2: trace riêng cho w = 'aab' (chỉ in các dòng của w[0]):")
import io, contextlib
buf = io.StringIO()
with contextlib.redirect_stdout(buf):            # gom output của trace rồi chỉ in các dòng của w[0]
    trace_hashmap(w2)
print("\n".join(line for line in buf.getvalue().splitlines() if line.startswith("w[0]")))

Bài 1(c): naive trả về [[1, 0], [0, 2], [0, 1], [0, 1], [2, 1], [1, 0]] -> 6 phần tử
Bài 2: trace riêng cho w = 'aab' (chỉ in các dòng của w[0]):
w[0]='aab'    j=0  |aab     A: tìm 'baa' -> có [1] => THÊM [1, 0]
w[0]='aab'    j=1  a|ab     A: tìm 'ba' -> không có
w[0]='aab'    j=2  aa|b     A: tìm 'b' -> có [2] => THÊM [2, 0]; B: tìm 'aa' -> không có
w[0]='aab'    j=3  aab|     B: j = k, bỏ qua


In [35]:
# ---------- Đáp án Bài 3 ----------
def exercise_palindrome_pairs(words):
    index = {w: i for i, w in enumerate(words)}
    result = []
    for i, w in enumerate(words):
        k = len(w)
        for j in range(k + 1):
            prefix, suffix = w[:j], w[j:]
            if prefix == prefix[::-1]:                       # Luật A
                x = suffix[::-1]
                if x in index and index[x] != i:
                    result.append([index[x], i])
            if j < k and suffix == suffix[::-1]:             # Luật B
                x = prefix[::-1]
                if x in index and index[x] != i:
                    result.append([i, index[x]])
    return result


assert same_pairs(exercise_palindrome_pairs(["abcd", "dcba", "lls", "s", "sssll"]), [[0, 1], [1, 0], [3, 2], [2, 4]])
assert same_pairs(exercise_palindrome_pairs(["bat", "tab", "cat"]), [[0, 1], [1, 0]])
assert same_pairs(exercise_palindrome_pairs(["a", ""]), [[0, 1], [1, 0]])
assert not has_duplicates(exercise_palindrome_pairs(["abcd", "dcba", "lls", "s", "sssll"]))
print("Bài 3 PASS")


# ---------- Đáp án Bài 4 ----------
def exercise_palindrome_pairs_trie(words):
    def is_pal(s, lo, hi):
        while lo < hi:
            if s[lo] != s[hi]:
                return False
            lo, hi = lo + 1, hi - 1
        return True

    root = [{}, -1, []]                                  # [children, word_id, pal_below]
    for e, x in enumerate(words):
        r, node = x[::-1], root
        for d in range(len(r)):
            if is_pal(r, d, len(r) - 1):
                node[2].append(e)
            node = node[0].setdefault(r[d], [{}, -1, []])
        node[1] = e
        node[2].append(e)

    res = []
    for i, w in enumerate(words):
        node = root
        for j, ch in enumerate(w):
            if node[1] not in (-1, i) and is_pal(w, j, len(w) - 1):
                res.append([i, node[1]])
            node = node[0].get(ch)
            if node is None:
                break
        else:
            res += [[i, e] for e in node[2] if e != i]
    return res


assert same_pairs(exercise_palindrome_pairs_trie(["abcd", "dcba", "lls", "s", "sssll"]), [[0, 1], [1, 0], [3, 2], [2, 4]])
assert same_pairs(exercise_palindrome_pairs_trie(["a", ""]), [[0, 1], [1, 0]])
assert same_pairs(exercise_palindrome_pairs_trie(["aba", "", "ab"]), [[0, 1], [1, 0], [2, 0]])
print("Bài 4 PASS")

Bài 3 PASS
Bài 4 PASS


In [36]:
# ---------- Đáp án Bài 5 ----------
from collections import defaultdict


def exercise_palindrome_pairs_with_duplicates(words):
    index = defaultdict(list)                            # từ -> list chỉ số
    for i, w in enumerate(words):
        index[w].append(i)
    result = []
    for i, w in enumerate(words):
        k = len(w)
        for j in range(k + 1):
            prefix, suffix = w[:j], w[j:]
            if prefix == prefix[::-1]:
                for t in index.get(suffix[::-1], []):     # .get: không tạo khóa rỗng mới
                    if t != i:
                        result.append([t, i])
            if j < k and suffix == suffix[::-1]:
                for t in index.get(prefix[::-1], []):
                    if t != i:
                        result.append([i, t])
    return result


assert same_pairs(exercise_palindrome_pairs_with_duplicates(["a", "a", ""]), palindrome_pairs_brute(["a", "a", ""]))
assert same_pairs(exercise_palindrome_pairs_with_duplicates(["ab", "ba", "ba"]), [[0, 1], [1, 0], [0, 2], [2, 0]])
assert exercise_palindrome_pairs_with_duplicates(["abc", "abc"]) == []

random.seed(5)                                           # đối chiếu với brute force khi có từ trùng
for _ in range(1000):
    ws = [random.choice(["", "a", "b", "ab", "ba", "aa", "aba"]) for _ in range(random.randint(1, 6))]
    got = exercise_palindrome_pairs_with_duplicates(ws)
    assert same_pairs(got, palindrome_pairs_brute(ws)) and not has_duplicates(got), ws
print("Bài 5 PASS (kể cả 1000 bộ ngẫu nhiên có từ trùng)")

Bài 5 PASS (kể cả 1000 bộ ngẫu nhiên có từ trùng)


In [37]:
# ---------- Đáp án Bài 6 (LeetCode 2131) ----------
def exercise_longest_palindrome_two_letters(words):
    cnt = Counter(words)
    length = 0
    has_center = False
    for w, c in cnt.items():
        if w[0] == w[1]:                                 # "xx": gương của chính nó
            length += (c // 2) * 4
            if c % 2 == 1:
                has_center = True
        elif w < w[::-1]:                                # "xy" với x < y: chỉ đếm một chiều
            length += min(c, cnt[w[::-1]]) * 4
    return length + (2 if has_center else 0)


assert exercise_longest_palindrome_two_letters(["lc", "cl", "gg"]) == 6
assert exercise_longest_palindrome_two_letters(["ab", "ty", "yt", "lc", "cl", "ab"]) == 8
assert exercise_longest_palindrome_two_letters(["cc", "ll", "xx"]) == 2
assert exercise_longest_palindrome_two_letters(["ab", "ba", "ab", "ba", "aa", "aa", "aa"]) == 8 + 4 + 2
print("Bài 6 PASS")

Bài 6 PASS


## 28. Template ngắn nên nhớ

**Tư duy:** palindrome `A + B` ⟹ từ dài hơn = *gương của từ ngắn hơn* + *khúc giữa palindrome*. Đứng ở mỗi từ, **cắt ở mọi vị trí**, tra "gương" trong hash map.

```python
index = {w: i for i, w in enumerate(words)}
for i, w in enumerate(words):
    for j in range(len(w) + 1):                 # k + 1 lát cắt
        p, s = w[:j], w[j:]
        if p == p[::-1] and index.get(s[::-1], i) != i:
            res.append([index[s[::-1]], i])     # gương(s) + w
        if j < len(w) and s == s[::-1] and index.get(p[::-1], i) != i:
            res.append([i, index[p[::-1]]])     # w + gương(p)
```

Ba "chốt an toàn": `range(len(w) + 1)` · `!= i` · `j < len(w)` ở luật thứ hai.

Bản Trie (khi cần `O(tổng độ dài)`): Trie trên **từ đảo ngược**, mỗi nút có `word_id` và `pal_below`; đi `w` xuống Trie: gặp `word_id` giữa đường → kiểm tra `w[j:]`; đi hết `w` → lấy cả `pal_below`. Thay kiểm tra palindrome bằng cờ từ `lps` của `w + "#" + reverse(w)`.

---

## 29. Checklist trước khi submit

- [ ] Dựng `dict` **từ → chỉ số** (không dùng `words.index`)?
- [ ] Lặp `j` trong `range(k + 1)` — có cả hai lát cắt với phần rỗng?
- [ ] Luật A thêm `[index[x], i]` (từ tìm được đứng **trước**), Luật B thêm `[i, index[x]]` (đứng **sau**)?
- [ ] Chặn `index[x] != i` để không ghép từ với chính nó (từ palindrome như `"aba"`, `"s"`)?
- [ ] Chặn **đúng một** trong hai lát cắt biên (`j < k` ở Luật B) để không trùng — không chặn cả hai (sẽ sót)?
- [ ] Đã thử `["a", ""]`, `[""]`, `["aba"]`, `["abcd", "dcba"]`, `["a", "aa", "aaa"]` và ba ví dụ của đề?
- [ ] Test bằng `sorted(...)` vì đáp án theo thứ tự bất kỳ, và kiểm tra không có cặp trùng?
- [ ] Độ phức tạp `O(n·k²)` với `n <= 5000`, `k <= 300` — chạy dưới vài giây; nếu bị hỏi `O(tổng độ dài)` thì trình bày được Trie + `lps`?